# Pickup grid

An experiment, not a tool. One cuboid in the dish, the robot picks it up over and
over at different combinations of two parameters, and the share of successful
pickups is recorded. It exists to check the pickup equation empirically; when the
numbers are in, the notebook has done its job.

**The two axes**

* `pickup_offset_mm` — the tip height **above the bottom of the dish**, not above
  the cuboid. It is `PickingConfig.pickup_offset`, and `pickup_height =
  dish_bottom + pickup_offset` is computed by the config itself.
* `aspirate_flow_rate` — the flow rate while aspirating, µl/s.

The third variable, the size of the cuboid, is the operator's: one size is one run
of this notebook under its own label.

**What counts as a success.** After the lift the dish is photographed. Empty means
the cuboid is in the tip. It is then put back and the dish photographed again —
and that second check is not optional. Without it a cuboid that never came back
(still in the tip, dispensed past the dish) leaves an empty dish on the next
iteration, which is scored a success although there was nothing to pick. So there
are three outcomes and not two: `success`, `miss` and `lost`, and a `lost` stops
the loop and calls the operator.

**What is deliberately absent.** `PickingSession` is not used here: no routine, no
plate, no floaters, no batching, no shape filters. The loop is written in this
notebook on top of `hardware/protocols` and `core/vision/cuboids`. Detection is
`detect_boxes` + `build_cuboid_df` + `add_derived` with **no** `select_pickable`:
there is one cuboid in the dish and nothing to filter, so the largest detection
inside the ROI is the cuboid.

**Statistics.** At ten attempts per cell the standard error of a proportion is
about 0.15 at p ≈ 0.5. That separates 0.2 from 0.8. It does **not** separate 0.5
from 0.7, and no amount of colour on a heat map will change that. The intervals
below are Wilson intervals, because the normal approximation is wrong at n = 10.

Nothing in `core/`, `hardware/`, `workflows/` or `config/` is touched by this
notebook.

## 0. Setup

`bench_setup` fixes `MICROPICK_ROOT` and re-exports the common names. It lives one
directory up, so it is found by walking upwards rather than by assuming where the
kernel was started.

In [ ]:
import csv
import json
import math
import re
import shutil
import subprocess
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import pandas as pd


def _notebooks_dir(start: Path) -> Path:
    """The directory holding bench_setup.py, searching upwards from `start`."""
    for d in (start, *start.parents):
        if (d / "bench_setup.py").is_file():
            return d
    raise RuntimeError(
        f"bench_setup.py not found above {start}; start the kernel inside the "
        f"repository")


sys.path.insert(0, str(_notebooks_dir(Path.cwd())))

from bench_setup import *                          # noqa: E402,F401,F403
from micropick.hardware.protocols import (move_relative, move_to,  # noqa: E402
                                          require_ok, xyz)

pd.set_option("display.width", 200)
print(paths.describe())

## 1. Constants

Everything that decides what this run is, in one cell — which now sits down by
the grid, next to the button that starts it. `EXPERIMENT_LABEL` names the results
folder and carries the cuboid size, which is the one variable the operator sets by
hand.

**Running the constants cell applies nothing on its own.** The numbers are read
when the rig is built, and nowhere else. To run the grid with different
parameters: edit the constants, run that cell, then run *Start, or carry on* —
which builds a fresh rig, opens a new results folder and prints, in one line,
what the rig will actually do. If a number you changed is not in that line, it
did not take effect.

The parameters here are the experiment's own. They are **not** `profile.picking`:
this notebook builds its own `PickingConfig` from these constants because
`core/vision` needs an object of that type, and it neither reads picking.json for
them nor writes anything back. Sharing that object is what made the first bench
runs aspirate 10 µl while the config printed 30.

**`ADAPTIVE_HEIGHTS`** decides where the heights come from: `True` finds them as
the run goes — each flow rate starting where the previous one was still fully
successful and climbing until a height scores nothing, between
`PICKUP_OFFSET_START_MM` and the safety ceiling `PICKUP_OFFSET_MAX_MM` — and
`False` runs `PICKUP_OFFSETS_MM` as listed. Section 10 has the rules.

**`ASPIRATE_VOLUME` and `ASPIRATE_TIME_S` decide what the flow-rate axis
means.** At a fixed volume the aspiration lasts `volume / flow`: 10 µl is a
second at 10 µl/s and a twentieth of a second at 200 µl/s, so moving along the
axis changes both how fast the medium moves and how long it moves for. With
`ASPIRATE_VOLUME = None` the volume is `flow_rate × ASPIRATE_TIME_S`, every
aspiration lasts the same time, and the axis is flow rate alone; at 1.0 s the
volume is numerically the flow rate. The rig prints which of the two it is
doing, `preflight` prints the volume and the times for every flow rate, and the
largest volume has to fit `TIP_CAPACITY_UL`. Switching modes changes the
conditions, so a folder started in one cannot be carried on in the other.

`RETURN_FLOW_RATE` is deliberately not taken from the grid. Dropping the cuboid
back at the grid's flow rate would hit it hydrodynamically before the next
attempt, and the flow rate would then be correlated with itself through the
history of the dish. The return is always the same.

`RETURN_OFFSET_MM` is there for the same reason and does the same job for the
other axis: the cuboid goes back at one fixed height above the dish bottom,
never at the height being tested. Together with the dish centre taught below,
that makes every return identical — same place, same height, same flow — so
nothing about the trip back varies with the parameter under test.

## 2. Connecting

Profile, robot, cameras, detector. Section 5 of `01_robot_session` has to have run
at some point: this needs a pixel map and a pipette offset in the profile, and a
taught `observe` position.

In [ ]:
PROFILE = "lab_main"

profile = load_profile(PROFILE)
profile.require_calibration()
pmap = PixelMap.from_config(profile.pixel_map)
off = profile.calibration.pipette_offset
tip_offset = np.array([off.dx, off.dy], dtype=float)
observe = np.array(profile.where("observe"), dtype=float)

print("profile:", profile.meta.name, "| schema", profile.meta.schema_version)
print(f"map: degree {pmap.config.degree}, holdout "
      f"{pmap.config.holdout_mean_um or float('nan'):.1f} um")
print(f"pipette offset: ({off.dx:.2f}, {off.dy:.2f}) mm, {off.method}")
print("observe:", tuple(round(v, 2) for v in observe))

In [ ]:
openapi = ot2_api.OpentronsAPI()
openapi.add_slot_offsets([5, 8, 9], (0, 0, 64.2))

In [ ]:
openapi.toggle_lights()

In [ ]:
r = openapi.get_run_info()

In [ ]:
# Once after power on.
openapi.home_robot()

In [ ]:
openapi.create_run()
openapi.load_pipette()
print("run:", openapi.run_id, " pipette:", openapi.pipette_id)

In [ ]:
labware.ensure_definitions(openapi)

TIP_RACK = "vwr_96_tiprack_200ul_xl"
labware.load_labware(openapi, TIP_RACK, 10)

In [ ]:
openapi.drop_tip_in_place()

In [ ]:
openapi.pick_up_tip(openapi.labware_dct["10"], "A2")

In [ ]:
openapi.retract_axis('leftZ')

In [ ]:
cams = open_cameras(profile)
over_cam = cams.open("overview_cam")
print(over_cam)

problems = profile.pixel_map.check_camera(over_cam.resolution)
require(not problems,
        "the calibration does not match the camera: " + "; ".join(problems))

# under_cam = (cams.open("underview_cam", resolution=(2000, 1500))
#              if RECORD_CLIPS else None)
# if under_cam is not None:
#     print(under_cam)

In [ ]:
from ultralytics import YOLO

CUBOID_WEIGHTS = "cuboid_bbox_v4-11_best.pt"
_weights = paths.ml_models_dir() / CUBOID_WEIGHTS
require(_weights.exists(),
        f"cuboid detector weights not found: {_weights}\n"
        f"put the .pt file in {paths.ml_models_dir()} "
        f"(weights are not tracked in the repository)")
cuboid_model = YOLO(str(_weights))
print("loaded", CUBOID_WEIGHTS)

## Tip calibration

In [ ]:
from ultralytics import YOLO
LIMITS = Limits(x=(0, 380), y=(0, 350), z=(0.1, 150))

tip_model = YOLO(str(paths.ml_models_dir() / profile.calibration.tip_target.model_file))
tip_detector = TipDetector(tip_model,
                           imgsz=profile.calibration.tip_target.imgsz,
                           conf=profile.calibration.tip_target.conf)
under_cam = cams.open("underview_cam")
print(under_cam)

In [ ]:
openapi.toggle_lights()

In [ ]:
openapi.retract_axis('leftZ')

In [ ]:
under_cam = cams.open("underview_cam")
print(under_cam)

target = profile.calibration.tip_target
openapi.move_to_coordinates(profile.where("tip_calib"),
                            min_z_height=target.module_height - 0.1, verbose=False)
time.sleep(0.5)

def touch_up(robot, camera, view):
    ctrl = JogController(robot, limits=LIMITS, step=0.05)
    jog_in_window(ctrl, camera, window="tip",
                  title="nudge the tip onto the crosshair, then Enter")

# The profile goes in, so the routine can check the lower camera is at the mode
# the calibration is defined at (its CameraSpec default) before anything moves,
# and so it saves the offset and the homography together - they are products of
# the same measurement, and saving them apart is how a profile ends up with one
# refreshed and the other left from an older tip.
current = profile.calibration.pipette_offset
result = calibrate_pipette_offset(
    openapi, over_cam, under_cam, tip_detector, pmap,
    target=target,
    current_offset=(current.dx, current.dy),
    frames=7,
    tip_type=current.tip_type,
    profile=profile,
    manual_touch_up=touch_up)          # pass None to skip the manual step

print()
print(result)
print("saved:", profile.calibration.pipette_offset)

openapi.retract_axis('leftZ')

### Check calibration

In [ ]:
profile = store.load_profile(PROFILE)
profile.require_calibration()
pmap = PixelMap.from_config(profile.pixel_map)
off = profile.calibration.pipette_offset
tip_offset = np.array([off.dx, off.dy])

problems = profile.pixel_map.check_camera(over_cam.resolution)
if problems:
    raise RuntimeError("the calibration does not match the camera: " + "; ".join(problems))

def pixel_to_robot(u, v, gantry_xy):
    """Robot coordinates that put the pipette tip on the pixel (u, v)."""
    if not pmap.covers(u, v):
        raise ValueError(f"pixel ({u:.0f}, {v:.0f}) is outside the calibrated area")
    return pmap.to_robot(u, v, gantry_xy) + tip_offset

def area_mm2(area_px, u, v):
    su, sv = pmap.mm_per_px(u, v)
    return area_px * su * sv

print("ready:", pmap.config.degree, "degree map,",
      f"holdout {pmap.config.holdout_mean_um:.1f} um,",
      f"offset ({off.dx:.2f}, {off.dy:.2f}) mm")

In [ ]:
def click_to_go(z=None, snap_px=60, conf=0.25, imgsz=2016, move=True):
    win = "click to go"
    state = {"dets": [], "click": None, "frame": None, "gantry": None}
    history = []

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            state["click"] = (x, y)

        if event == cv2.EVENT_RBUTTONDOWN:
            openapi.move_to_coordinates(profile.where("tip_calib"))

    def detect_now():
        g = np.array(xyz(openapi)[:2])
        frame = over_cam.read_after(time.monotonic())
        res = tip_model.predict(source=frame[..., ::-1], conf=conf, imgsz=imgsz,
                                save=False, verbose=False)
        pts = []
        for r in res:
            for b in r.boxes:
                if tip_model.names[int(b.cls[0])] != "point":
                    continue
                x1, y1, x2, y2 = (float(v) for v in b.xyxy[0])
                p = np.array([(x1 + x2) / 2, (y1 + y2) / 2])
                if pmap.covers(*p):
                    pts.append((p, pmap.to_robot(p[0], p[1], g)))
        state["dets"], state["gantry"] = pts, g
        return pts

    print("детекция...")
    detect_now()
    print(f"найдено {len(state['dets'])} крестов")

    view = FrameWindow(win)
    view.fit(over_cam.read()[1].shape)   # so the mouse callback has a window
    cv2.setMouseCallback(win, on_mouse)
    try:
        while True:
            ok, frame = over_cam.read()
            if not ok:
                continue
            vis = frame.copy()
            h, w = vis.shape[:2]
            cv2.drawMarker(vis, tuple(np.int32(pmap.config.ref)), (0, 0, 255),
                           cv2.MARKER_CROSS, 60, 2)
            for p, world in state["dets"]:
                cv2.circle(vis, tuple(np.int32(p)), 14, (0, 255, 0), 2)
            cv2.putText(vis, f"{len(state['dets'])} crosses   click one   "
                             f"d=redetect  r=reset  Esc=quit", (20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
            if history:
                e = np.array(history)
                cv2.putText(vis, f"map consistency: mean {e.mean()*1000:.0f} um, "
                                 f"max {e.max()*1000:.0f} um  (n={len(e)})",
                            (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 1.2,
                            (0, 200, 255), 2)
            view.show(vis)

            key = cv2.waitKey(20) & 0xFF
            if key == 27:
                break
            if key == ord("d"):
                detect_now(); print(f"найдено {len(state['dets'])}")
            if key == ord("r"):
                history.clear()

            if state["click"] is None:
                continue
            cx, cy = state["click"]
            state["click"] = None
            if not state["dets"]:
                print("нет детекций, нажми d"); continue

            # привязка к ближайшему кресту в координатах отображаемого кадра
            # scale_x = frame.shape[1] / cv2.getWindowImageRect(win)[2]
            # scale_y = frame.shape[0] / cv2.getWindowImageRect(win)[3]
            # click_px = np.array([cx * scale_x, cy * scale_y])
            click_px = np.array([cx, cy])
            d = [np.linalg.norm(p - click_px) for p, _ in state["dets"]]
            k = int(np.argmin(d))
            # if d[k] > snap_px * max(scale_x, scale_y):
            #     print(f"мимо креста ({d[k]:.0f} px до ближайшего)"); continue
            if d[k] > snap_px:
                print(f"мимо креста ({d[k]:.0f} px до ближайшего)"); continue

            px, world = state["dets"][k]
            target = world + tip_offset
            print(f"\nкрест на пикселе ({px[0]:.0f}, {px[1]:.0f})")
            print(f"  координата креста : {world.round(3)}")
            print(f"  цель для пипетки  : {target.round(3)}")
            if not move:
                continue

            goto_xy(openapi, target[0], target[1])
            if z is not None:
                openapi.move_to_coordinates((target[0], target[1], z), min_z_height=1, verbose=False)
                # move_to(openapi, (target[0], target[1], z))

            # тот же крест заново, из новой позы
            after = detect_now()
            if after:
                worlds = np.array([wr for _, wr in after])
                j = int(np.argmin(np.linalg.norm(worlds - world, axis=1)))
                drift = float(np.linalg.norm(worlds[j] - world))
                history.append(drift)
                print(f"  тот же крест из новой позы: {worlds[j].round(3)}")
                print(f"  расхождение карты: {drift*1000:.0f} um")
    finally:
        view.close()

    if history:
        e = np.array(history)
        print(f"\nсогласованность карты по {len(e)} переездам: "
              f"среднее {e.mean()*1000:.0f} мкм, максимум {e.max()*1000:.0f} мкм")
    return history



In [ ]:
drift = click_to_go(z = 67.0)

In [ ]:
openapi.retract_axis('leftZ')

### The dish: its bottom and its centre

Both numbers come out of **one** pose. Jog the tip to the middle of the dish and
bring it down until it just touches the bottom: the z of that pose is
`dish_bottom`, and its x and y are the point every cuboid is returned to.

Arrows or WASD move x and y, `q` and `e` move z, `+` and `-` change the step,
Enter finishes. The window has to have focus, so a stray keystroke in the
notebook cannot drive the robot. Come down in decreasing steps — 0.5, then 0.1 —
and stop at the touch; the number wanted is where the tip meets the glass, not
where it pushes into it.

**Why the return goes to the centre** and not to where the cuboid was picked up
from: a cuboid put back where it came from wanders. Each attempt leaves it a
little further from where it started, and after two hundred attempts it is
against the rim, in a different depth of medium, in a different part of the frame
and at a different local scale of the pixel map. Returning to one taught point
keeps the dish in the state the run started in. It also makes the returns
identical to each other, which is the argument behind `RETURN_FLOW_RATE` and
`RETURN_OFFSET_MM` applied to the third coordinate.

**Teach it once, before the first trial, and not again during the run.**
`dish_bottom` is the origin of the height axis: moving it halfway through leaves
one half of the table in different coordinates from the other, which is why a
resumption whose `dish_bottom` disagrees with `run.json` is refused rather than
carried on with.

In [ ]:
def jog(title="", camera=None, step=1.0):
    """Manual control from a window, with the camera live in it."""
    ctrl = JogController(openapi, limits=LIMITS, step=step)
    pos = jog_in_window(ctrl, camera or over_cam, title=title)
    print("stopped at", tuple(round(v, 2) for v in pos))
    return pos


def teach_dish(step: float = 0.5) -> np.ndarray:
    """Jog to the middle of the dish, touch the bottom, keep both numbers.

    The pose goes into the profile as `dish_center`, all three coordinates: its
    xy is the point cuboids are returned to, its z is the dish bottom. That is
    the only place either number is kept, and it is where `make_rig` reads them
    from, so a kernel restart does not mean touching the glass again and no
    notebook variable can disagree with it.

    Nothing is written to picking.json. This is one experiment's dish, and the
    profile's picking parameters belong to the picking session.
    """
    x, y, z = (float(v) for v in
               jog("centre of the dish, tip on the bottom, then Enter",
                   step=step))
    profile.remember("dish_center", (x, y, z))
    # Off the glass straight away: the number is recorded, and nothing good
    # comes of leaving the tip resting on the bottom while this is read.
    move_relative(openapi, "z", 5.0)
    print(f"dish bottom {z:.2f} mm, centre ({x:.2f}, {y:.2f})")
    print("built rigs do not update themselves; run the rig cell again")
    return np.array([x, y, z])


def load_dish() -> np.ndarray:
    """The pose taught earlier, out of the profile. No robot motion."""
    x, y, z = profile.where("dish_center")
    print(f"dish bottom {z:.2f} mm, centre ({x:.2f}, {y:.2f}) — from the profile")
    return np.array([float(x), float(y), float(z)])

In [ ]:
openapi.toggle_lights()

## 3. Looking at the dish

One frame, one detection pass, the largest object inside the ROI. No
`select_pickable`: there is a single cuboid in the dish, so a shape window can
only ever throw away the one thing being measured.

The gantry pose is read next to the frame the pixel came from, because the pose is
part of the conversion and not a correction applied afterwards.

`park` belongs here for the same reason: every frame of the experiment is taken
from the one observation pose, so that a pixel in one trial means what it meant in
the last.

In [ ]:
def look(rig):
    """One frame -> (frame, detections, gantry_xy). Nothing is filtered out."""
    frame = rig.camera.read_after(time.monotonic())
    gantry = np.array(xyz(rig.robot)[:2])
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    roi = vision.roi_mask(gray, rig.cfg, rig.one_d_ratio)
    boxes, confs = vision.detect_boxes(rig.detector, frame, rig.cfg)
    df = vision.build_cuboid_df(
        gray, boxes, confs, roi_mask=roi,
        pad=rig.cfg.otsu_pad, open_k=rig.cfg.otsu_open_k,
        bubble_core_r=rig.cfg.bubble_core_r,
        bubble_ring_window=rig.cfg.bubble_ring_window,
        bubble_min_area_px=rig.cfg.bubble_min_area_px)
    df = vision.add_derived(df, rig.size_ratio, rig.one_d_ratio,
                            rig.cfg.circle_center)
    return frame, df, gantry


def largest(df):
    """The biggest detection, or None on an empty dish."""
    if len(df) == 0:
        return None
    return df.loc[df.area.idxmax()]


def park(rig) -> None:
    """Take up the observation pose and let the dish settle.

    Every frame in this experiment is taken from here, so that the pixels of one
    trial mean the same as the pixels of the next.

    The move is skipped when the gantry is already there. Not only for speed: the
    robot declines an absolute move to the height a long tip already sits at
    (DESIGN section 9), and there is no reason to ask it for one.
    """
    if float(np.linalg.norm(np.array(xyz(rig.robot)) - rig.observe)) > 0.2:
        move_to(rig.robot, rig.observe, min_z_height=rig.cfg.dish_bottom,
                force_direct=True)
    time.sleep(rig.settle_s)

## 4. The rig

Everything a trial needs, passed in rather than read from globals, so the same
functions run against mocks in the dry run at the end.

In [ ]:
class TrialError(RuntimeError):
    """The trial could not be carried out at all, as opposed to failing."""


class GridAborted(RuntimeError):
    """The run stopped and wants a person, not a fix in code."""


@dataclass
class Rig:
    """The hardware and the numbers one trial works against.

    None of the numbers has a default, and that is the point. They used to:
    `volume: float = ASPIRATE_VOLUME` and three more like it. A dataclass default
    is evaluated once, when the class statement runs, so those fields held
    whatever the constants happened to be at that moment and never looked at them
    again — while `make_rig` passed none of them. Editing the constants cell and
    re-running it therefore changed nothing: every run on disk went out at 10 ul
    with a config on screen that said 30. The same trap DESIGN section 6
    describes for `pickup_height`.

    A field with no default cannot go stale. It has to be supplied on every
    construction, and `make_rig` reads the constants at the moment it is called.
    """

    robot: object
    camera: object
    detector: object
    cfg: object                      # a PickingConfig built for this run
    pmap: object
    offset: np.ndarray               # pipette offset, mm
    observe: np.ndarray              # the observation pose, xyz
    dish_center: np.ndarray          # taught xy the cuboid is returned to
    one_d_ratio: float               # mm per pixel at the dish centre
    size_ratio: float                # mm^2 per pixel^2 there
    settle_s: float                  # pause before a check frame
    volume: object                   # ul aspirated and returned, or None:
                                     # then it follows the flow rate
    aspirate_time_s: object          # s; with volume None, volume = flow x this
    return_flow: float               # ul/s on the way back
    return_offset: float             # mm above the bottom on the way back
    dip_wait_s: float                # s before dipping a tip that kept the cuboid
    recorder: object = None          # a hardware.camera.Recorder, or None
    clip_dir: object = None
    clip_crop: tuple = (0, 0)        # origin of what the recorder stores
    clip_size: tuple = (0, 0)
    under_cam: object = None         # the camera the recorder is attached to
    homography: object = None
    keep_clip: object = None         # callable(row) -> bool
    log: object = print

    def __post_init__(self):
        if self.volume is None and not (self.aspirate_time_s
                                        and self.aspirate_time_s > 0):
            raise ValueError(
                "ASPIRATE_VOLUME is None, so the volume follows the flow rate, "
                "and that needs ASPIRATE_TIME_S > 0")

    def volume_for(self, flow_rate: float) -> float:
        """How much one trial at this flow rate aspirates, and returns.

        The one place the rule lives. A fixed volume means the aspiration lasts
        volume / flow, which at 10 ul is a second at 10 ul/s and a twentieth of a
        second at 200 ul/s, so the flow-rate axis is really two things at once:
        how fast the medium moves and how long it moves for. With `volume` None
        the volume is `flow_rate * aspirate_time_s`, every aspiration lasts the
        same time, and the axis is flow rate alone. The return does not follow:
        it stays at `return_flow`, so its duration varies, but the return is not
        the part being measured.
        """
        if self.volume is not None:
            return float(self.volume)
        return float(flow_rate) * float(self.aspirate_time_s)

    @property
    def aspirate_mode(self) -> str:
        if self.volume is not None:
            return f"{float(self.volume):g} ul at every flow"
        return (f"{float(self.aspirate_time_s):g} s x flow "
                f"(volume = flow rate x {float(self.aspirate_time_s):g})")

    def summary(self) -> str:
        """What this rig will actually do, in one line.

        Printed whenever a rig is built, so a number that did not take effect is
        visible at once instead of two hundred trials later in run.json.
        """
        c = self.cfg
        cx, cy = self.dish_center
        return (f"rig: aspirate {self.aspirate_mode} | return {self.return_flow:g}"
                f" ul/s at ({cx:.2f}, {cy:.2f}), "
                f"{c.dish_bottom + self.return_offset:.2f} mm | settle "
                f"{self.settle_s:g} s | dish bottom {c.dish_bottom:.2f} mm | "
                f"lift {c.lift_mm:g} mm | ROI r={c.circle_radius} px about "
                f"{tuple(c.circle_center)} | detector conf {c.yolo_conf:g}, "
                f"imgsz {c.yolo_imgsz} | dip after {self.dip_wait_s:g} s")


def experiment_config(dish_bottom: float, **over) -> PickingConfig:
    """A PickingConfig for this experiment, built from the constants below.

    `core/vision` speaks PickingConfig — `detect_boxes`, `roi_mask` and
    `build_cuboid_df` all read one — so an object of that type is needed. It is
    deliberately not the *profile's* one. Only a handful of its fields mean
    anything here, the rest belong to the picking session, and sharing the object
    cost twice over: editing it edited the parameters a production run would use,
    and anything that reloads the profile (the calibration check above does)
    swapped the object underneath the experiment. One run on disk records a dish
    bottom that came back from picking.json rather than the one taught.

    Everything not named here stays at the schema default, which is what the
    detection fields in picking.json are set to anyway. To change one for a run,
    pass it: `make_rig(cfg=experiment_config(dish_bottom, yolo_conf=0.4))`.
    """
    # `vol` is left at the schema default on purpose: the volume belongs to the
    # rig and to the trial (`Rig.volume_for`), it can differ per cell, and
    # nothing in this notebook reads it off the config.
    base = dict(dish_bottom=float(dish_bottom),
                lift_mm=LIFT_MM, circle_center=ROI_CENTER,
                circle_radius=ROI_RADIUS, minimum_distance=MIN_DISTANCE_MM,
                clip_max_frames=CLIP_MAX_FRAMES)
    base.update(over)
    return PickingConfig(**base)


def make_rig(**over) -> Rig:
    """The bench rig, from the constants and the profile as they are *now*.

    Everything is read at call time: the constants cell, the taught dish, the
    pipette offset, the pixel map and the observation pose. So the way to run the
    grid with other parameters is edit the constants, run that cell, run this
    one — and the way to pick up a fresh tip calibration is to run this one
    again. Nothing carries over from the last rig.

    The dish comes out of the profile rather than out of a notebook variable,
    because `teach_dish` writes it there: one place holds it, and it survives a
    kernel restart. Its z is the dish bottom and its xy is the return point.
    """
    x, y, z = profile.where("dish_center")
    cfg = over.pop("cfg", None) or experiment_config(float(z))

    off = profile.calibration.pipette_offset
    pixel_map = PixelMap.from_config(profile.pixel_map)
    # The scale is taken at the dish centre and nowhere else. The pixel map is a
    # degree-3 polynomial, so asked outside its calibrated bounds it extrapolates
    # and answers with a plausible wrong number rather than refusing.
    mmpp = float(np.mean(pixel_map.mm_per_px(*cfg.circle_center)))

    base = dict(robot=openapi, camera=over_cam, detector=cuboid_model, cfg=cfg,
                pmap=pixel_map,
                offset=np.array([off.dx, off.dy], dtype=float),
                observe=np.array(profile.where("observe_2"), dtype=float),
                dish_center=np.array([float(x), float(y)]),
                one_d_ratio=mmpp, size_ratio=mmpp * mmpp,
                settle_s=SETTLE_S, volume=ASPIRATE_VOLUME,
                aspirate_time_s=ASPIRATE_TIME_S,
                return_flow=RETURN_FLOW_RATE, return_offset=RETURN_OFFSET_MM,
                dip_wait_s=RETURN_DIP_WAIT_S)
    base.update(over)
    rig = Rig(**base)
    print(rig.summary())
    return rig

## 5. Results on disk

`outputs/experiments/pickup_grid/<timestamp>_<label>/` holds `trials.csv`,
`trials.xlsx`, `run.json` and `clips/`.

**The CSV is the source of truth, not the xlsx.** A row cannot be appended to an
xlsx; the whole file is rewritten, and an interruption halfway through leaves a
corrupt archive. A CSV is appended a line at a time, so an interrupted run leaves
valid data behind. The xlsx is generated from it, at the end and on demand, so the
operator gets both.

`run.json` is the snapshot of the conditions. Without it the numbers are
unreadable in six months: which profile, which dish bottom, which tip offset,
which weights, which commit.

In [ ]:
RESULTS_ROOT = paths.outputs_dir() / "experiments" / "pickup_grid"

COLUMNS = ["trial_index", "cell_index", "flow_rate", "volume_ul",
           "pickup_offset_mm",
           "replicate", "outcome", "diameter_microns", "area_px", "cX", "cY",
           "target_x", "target_y", "target_z", "timestamp", "return_dips",
           "clip_path", "note"]

OUTCOMES = ("success", "miss", "lost", "skipped")


def append_row(run_dir, row: dict) -> None:
    """One line onto trials.csv; the header goes in when the file is created.

    An existing file has to carry exactly these columns. `volume_ul` was added
    after the first runs, and a row with one more field appended under an older
    header would be read back shifted by one column, silently.
    """
    path = Path(run_dir) / "trials.csv"
    fresh = not path.exists()
    if not fresh:
        with open(path, newline="", encoding="utf-8") as fh:
            header = next(csv.reader(fh), [])
        if header != COLUMNS:
            raise GridAborted(
                f"{path.name} in {path.parent.name} has an older column set "
                f"({len(header)} columns, this notebook writes {len(COLUMNS)}). "
                f"Nothing was written. Start a new run folder; the old one stays "
                f"readable by the analysis section.")
    with open(path, "a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=COLUMNS, extrasaction="raise")
        if fresh:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in COLUMNS})


def read_trials(run_dir) -> list:
    """Every row written so far, in order. [] when there is no file yet."""
    path = Path(run_dir) / "trials.csv"
    if not path.exists():
        return []
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def write_xlsx(run_dir):
    """Generate trials.xlsx from the CSV. Safe to call at any time."""
    run_dir = Path(run_dir)
    df = pd.read_csv(run_dir / "trials.csv")
    out = run_dir / "trials.xlsx"
    stats = cell_stats(df)
    with pd.ExcelWriter(out, engine="openpyxl") as xl:
        df.to_excel(xl, sheet_name="trials", index=False)
        stats.to_excel(xl, sheet_name="cells", index=False)
        rate_table(stats).to_excel(xl, sheet_name="success_rate")
    return out

In [ ]:
# The summary the xlsx and the analysis section both rest on. It lives here, next
# to the writing, because run_grid builds the xlsx when it finishes and the two
# would otherwise be defined after the cell that needs them.

def wilson(k: int, n: int, z: float = 1.96) -> tuple:
    """Wilson score interval. At n = 10 the normal approximation is wrong: it puts
    bounds outside [0, 1] and gives a zero-width interval at p = 0 or p = 1."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, centre - half), min(1.0, centre + half)


def cell_stats(df: pd.DataFrame) -> pd.DataFrame:
    """One row per grid cell: counts, rate, and the Wilson interval.

    A `lost` trial is counted and shown but left out of the rate: the cuboid never
    came back, so whether it was picked up says nothing about the parameters.
    """
    rows = []
    for (flow, offset), g in df.groupby(["flow_rate", "pickup_offset_mm"]):
        n_success = int((g.outcome == "success").sum())
        n_miss = int((g.outcome == "miss").sum())
        scored = n_success + n_miss
        low, high = wilson(n_success, scored)
        rows.append(dict(flow_rate=float(flow), pickup_offset_mm=float(offset),
                         skipped=bool((g.outcome == "skipped").any()),
                         n_success=n_success, n_miss=n_miss,
                         n_lost=int((g.outcome == "lost").sum()),
                         n_scored=scored,
                         rate=(n_success / scored if scored else float("nan")),
                         ci_low=low, ci_high=high))
    return (pd.DataFrame(rows)
            .sort_values(["flow_rate", "pickup_offset_mm"])
            .reset_index(drop=True))


def rate_table(stats: pd.DataFrame) -> pd.DataFrame:
    """Success rate, heights down the side, flow rates across. NaN = no data."""
    return stats.pivot(index="pickup_offset_mm", columns="flow_rate",
                       values="rate")

In [ ]:
def repo_commit() -> dict:
    """The commit this ran at, when the repository is available."""
    def git(*args):
        return subprocess.run(["git", "-C", str(paths.root()), *args],
                              capture_output=True, text=True,
                              timeout=5).stdout.strip()
    try:
        return {"commit": git("rev-parse", "HEAD") or None,
                "dirty": bool(git("status", "--porcelain"))}
    except Exception:
        return {"commit": None, "dirty": None}


def run_snapshot(rig, *, label, offsets, flow_rates, trials, weights) -> dict:
    """The conditions of this run, as they go into run.json."""
    import ultralytics

    c = rig.cfg
    po = profile.calibration.pipette_offset
    dx, dy = (float(v) for v in rig.offset)
    return {
        "label": label,
        "cuboid_size_um": CUBOID_SIZE_UM,
        "started_at": datetime.now().isoformat(timespec="seconds"),
        "profile": {"name": profile.meta.name,
                    "schema_version": profile.meta.schema_version},
        # the five fields a resumption is checked against, from the one function
        # that states them (defined with run_grid, which compares them)
        **run_conditions(rig, offsets=offsets, flow_rates=flow_rates,
                         trials=trials),
        # derived from the rule and the grid, written out so the file reads
        # without arithmetic: what was drawn at each flow rate, and for how long
        "volume_by_flow_ul": {f"{f:g}": round(v, 3)
                              for f, v, _, _ in volume_table(rig, flow_rates)},
        "aspirate_seconds_by_flow": {f"{f:g}": round(t, 3)
                                     for f, _, t, _ in volume_table(rig, flow_rates)},
        "return_dip_wait_s": rig.dip_wait_s,
        "dish_center": [float(v) for v in rig.dish_center],
        "pipette_offset": {"dx": dx, "dy": dy, "tip_type": po.tip_type,
                           "method": po.method,
                           "measured_at": (po.measured_at.isoformat()
                                           if po.measured_at else None)},
        "settle_s": rig.settle_s,
        "lift_mm": c.lift_mm,
        "detector": {"weights": weights,
                     "ultralytics": ultralytics.__version__,
                     "imgsz": c.yolo_imgsz, "conf": c.yolo_conf,
                     "iou": c.yolo_iou, "max_det": c.yolo_max_det},
        "vision": {"circle_center": list(c.circle_center),
                   "circle_radius": c.circle_radius,
                   "otsu_pad": c.otsu_pad, "otsu_open_k": c.otsu_open_k,
                   "mm_per_px_at_centre": rig.one_d_ratio},
        # off the rig, not off a notebook global: the rig holds the map, the
        # offset and the dish it will actually use, and a stale global is exactly
        # what this snapshot exists to catch
        "pixel_map": {"degree": rig.pmap.config.degree,
                      "holdout_mean_um": rig.pmap.config.holdout_mean_um,
                      "image_size": list(rig.pmap.config.image_size),
                      "fitted_at": (rig.pmap.config.fitted_at.isoformat()
                                    if rig.pmap.config.fitted_at else None)},
        "clips": {"enabled": rig.recorder is not None,
                  "max_frames": c.clip_max_frames},
        "repo": repo_commit(),
    }


# What must not have changed between the start of a run and a resumption of it.
# dish_bottom is in the list because it is the origin of the height axis: shift it
# halfway through and the two halves of the table are in different coordinates.
#
# dish_center is recorded but deliberately not enforced. It is a pose touched by
# hand, so re-teaching it reproduces it to a few hundredths and no further, and
# the difference does not enter any measurement: the pickup point is detected
# afresh every trial, and the centre only says where the cuboid is put back. A
# dish that really moved shows up as a changed dish_bottom, which is enforced.
MUST_MATCH = ["label", "grid", "dish_bottom", "volume_ul", "aspirate_time_s",
              "return_flow_rate", "return_offset_mm"]


def check_snapshot(run_dir, snapshot: dict) -> dict:
    """Write run.json, or refuse a resumption whose conditions have changed."""
    path = Path(run_dir) / "run.json"
    if not path.exists():
        path.write_text(json.dumps(snapshot, indent=2) + "\n", encoding="utf-8")
        return snapshot

    stored = json.loads(path.read_text(encoding="utf-8"))
    differences = [f"  {key}: run.json has {stored.get(key)!r}, "
                   f"now {snapshot.get(key)!r}"
                   for key in MUST_MATCH if stored.get(key) != snapshot.get(key)]
    if differences:
        raise GridAborted(
            "this folder was started under different conditions:\n"
            + "\n".join(differences)
            + "\nCarrying on would mix two experiments in one table. Either "
              "restore the values above, or start a new run folder.")
    return stored


def open_run_dir(resume=None):
    """A fresh results folder, or an existing one to be carried on."""
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    if resume:
        run_dir = RESULTS_ROOT / resume
        require(run_dir.is_dir(), f"no such run folder: {run_dir}")
    else:
        run_dir = RESULTS_ROOT / (time.strftime("%Y%m%d_%H%M%S")
                                  + f"_{EXPERIMENT_LABEL}")
        run_dir.mkdir(parents=True, exist_ok=False)
    (run_dir / "clips").mkdir(exist_ok=True)
    return run_dir


def previous_runs(label: str | None = None) -> list:
    """Folders for this label that already hold trials, oldest first.

    The label is resolved when this is called, not when it is defined. As a
    default argument it was evaluated once at definition time, which froze
    whatever EXPERIMENT_LABEL happened to be then — and with the constants cell
    now below this one, it could not even be defined on a clean run.
    """
    label = label or EXPERIMENT_LABEL
    if not RESULTS_ROOT.is_dir():
        return []
    return sorted(d for d in RESULTS_ROOT.glob(f"*_{label}")
                  if (d / "trials.csv").exists())

## 6. Clips

Off by default, but the plumbing is here in full, on the model of the
lower-camera recording in `01_robot_session`.

A clip cannot be started once a pickup has gone badly: by the time the outcome is
known the filming is over. So **every pickup goes into a ring buffer and the
buffer is thrown away on success**. `Recorder` is that ring buffer already — a
`deque` bounded by `PickingConfig.clip_max_frames` — and it keeps its frames after
`stop()` until the next `start()`, which is what lets the decision wait for the
confirmation frame two moves later.

What gets saved is one function, so the rule can change without touching the loop.

In [ ]:
def should_keep_clip(row: dict) -> bool:
    """Keep failures and losses only, by default."""
    return row["outcome"] in ("miss", "lost")


def clip_view(camera):
    """(origin, size, transform) for what the recorder should store.

    The same three-in-one as workflows/picking._clip_view: the origin subtracted
    from a point drawn on the clip and the crop applied to the frame have to be
    the same thing, and a crop of 1.0 means the whole frame.
    """
    frac = float(getattr(camera, "crop", 1.0))
    w, h = camera.resolution
    if frac >= 1.0:
        return (0, 0), (w, h), None
    x0, y0, side = vision.center_crop_box((h, w), frac)
    return (x0, y0), (side, side), lambda f: vision.center_crop(f, frac)[0]


def attach_recorder(rig, camera, run_dir, homography=None):
    """Give the rig a recorder on `camera`, writing into run_dir/clips.

    `homography` is the profile's `CameraHomography` when there is one. It is
    passed in rather than read off the profile so that the recording can be
    exercised with no profile at all.
    """
    origin, size, transform = clip_view(camera)
    rig.recorder = camera.record(max_frames=rig.cfg.clip_max_frames,
                                 transform=transform)
    rig.under_cam = camera
    rig.clip_dir = Path(run_dir) / "clips"
    rig.clip_dir.mkdir(parents=True, exist_ok=True)
    rig.clip_crop, rig.clip_size = origin, size
    rig.homography = (Homography.from_config(homography)
                      if homography is not None else None)
    if rig.homography is None:
        rig.log("no upper-to-lower homography: the clips record without a box "
                "on the cuboid, which is not an error")
    return rig


def mark_target(rig, uv, gantry) -> None:
    """Box the cuboid on the clip, if the profile can place one.

    Never fatal and never silent: the box is a viewing aid, and every way of not
    getting one says so, because failing quietly here is how it once went missing
    for a whole run of the machine.
    """
    if rig.recorder is None:
        return
    rig.recorder.clear_roi()
    if rig.homography is None:
        return
    try:
        drift = rig.homography.drift_mm(gantry)
        if drift > rig.cfg.homography_drift_warn_mm:
            rig.log(f"    homography was fitted {drift:.1f} mm from this pose; "
                    f"the box is approximate")
        under = rig.homography.over_to_under(
            np.array([uv]), gantry, rig.camera.resolution,
            rig.under_cam.resolution)
        mark = under - np.array(rig.clip_crop, dtype=float)
        w, h = rig.clip_size
        if not (0 <= mark[0, 0] < w and 0 <= mark[0, 1] < h):
            rig.log("    the ROI box falls outside the clip frame; check that "
                    "the homography was fitted on this disc and this camera")
        rig.recorder.mark_rois(mark)
    except Exception as exc:                    # a clip is never critical
        rig.log(f"    no ROI box on the clip: {exc}")


def estimate_clip_bytes(camera, cfg, n_trials: int) -> float:
    """The worst case: every trial fails and every clip is kept.

    The per-frame figure is a rough H.264 estimate and not a measurement — about
    0.1 byte per pixel at these resolutions and this much motion. It is here to
    tell 2 GB from 200 GB before the run starts, not to be accurate.
    """
    _, (w, h), _ = clip_view(camera)
    return n_trials * cfg.clip_max_frames * w * h * 0.1

## 7. Pre-start check

The dish has to show exactly one cuboid. Zero means there is nothing to pick; more
than one means the largest-object rule is choosing one of several, and every
number after that is about an unknown object. Both are refused rather than warned
about.

**The picture comes before the refusal.** What has to be fixed is in the dish, and
whoever fixes it needs to see where: a count on its own says a second object
exists but not that it is a bubble at the rim, or a chip of debris the ROI will
let in the moment the dish is nudged. So the frame is drawn and shown first, with
every detection the model returned in red, the ones that reached the table in
green, and the ROI as a circle — and only then is the run refused.

It also says how long the run will take and, with clips on, how much disk the
worst case needs.

In [ ]:
def show_detections(rig, frame, df, boxes) -> None:
    """The frame with everything that was found drawn on it.

    Red boxes are every detection the model returned, the ROI included or not;
    green contours are the objects that survived into the table. The circle is
    the ROI as `roi_mask` builds it, the working radius grown by the minimum
    spacing, so it is the boundary the table was actually filtered on.
    """
    vis = frame.copy()
    roi_px = (rig.cfg.circle_radius
              + int(rig.cfg.minimum_distance / rig.one_d_ratio))
    overlays.draw_dish(vis, rig.cfg.circle_center, roi_px)
    for x1, y1, x2, y2 in np.asarray(boxes, dtype=float).reshape(-1, 4):
        cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)),
                      (0, 0, 255), 2)
    overlays.draw_contours(vis, df, (0, 255, 0), 2)
    for n, (_, r) in enumerate(df.iterrows(), start=1):
        cv2.putText(vis, f"{n}: {r.diameter_microns:.0f} um",
                    (int(r.cX) + 14, int(r.cY) - 14), cv2.FONT_HERSHEY_SIMPLEX,
                    1.1, (0, 255, 0), 2)
    if vis.shape[1] > 1400:
        vis = cv2.resize(vis, (1400, int(1400 * vis.shape[0] / vis.shape[1])))
    show(vis, "red: every detection.  green: inside the ROI.  circle: the ROI")


def preflight(rig, *, offsets, flow_rates, trials) -> dict:
    """Show what the detector sees, then refuse unless it is exactly one cuboid.

    The picture and the table come before the refusal, and deliberately so: the
    reason to stop is nearly always something in the dish, and it cannot be taken
    out by someone who has only been told a count.
    """
    park(rig)
    t0 = time.monotonic()
    frame, df, gantry = look(rig)
    t_detect = time.monotonic() - t0

    # Everything the model returned, whether or not it survived the ROI and the
    # contour step. One extra inference on the frame already in hand: an object
    # just outside the working circle is not a reason to refuse, but it is very
    # much a reason to look, and this runs once.
    boxes, _ = vision.detect_boxes(rig.detector, frame, rig.cfg)

    print(f"detector: {len(boxes)} box(es), {len(df)} of them inside the ROI "
          f"and contoured, {t_detect:.2f} s per frame")
    for n, (_, r) in enumerate(df.iterrows(), start=1):
        print(f"  {n}: ({r.cX:7.1f}, {r.cY:7.1f})  area {r.area:8.0f} px  "
              f"{r.diameter_microns:6.0f} um  conf {r.conf:.2f}")
    if len(boxes) > len(df):
        print(f"  {len(boxes) - len(df)} detection(s) did not reach the table: "
              f"outside the ROI, or no usable contour. They are in red below.")
    show_detections(rig, frame, df, boxes)

    require(len(df) == 1,
            f"the experiment needs exactly one cuboid in the dish, and the "
            f"detector sees {len(df)} inside the ROI ({len(boxes)} in the whole "
            f"frame). "
            + ("Put one in." if len(df) == 0 else
               "Take the others out, or move the ROI so that only one is in it.")
            + " The picture above is the state that was refused.")

    obj = df.iloc[0]
    require(rig.pmap.covers(float(obj.cX), float(obj.cY)),
            "the cuboid is outside the calibrated area of the pixel map")

    cx, cy = rig.dish_center
    print(f"\ndish bottom {rig.cfg.dish_bottom:.2f} mm; the cuboid goes back to "
          f"({cx:.2f}, {cy:.2f}) at "
          f"{rig.cfg.dish_bottom + rig.return_offset:.2f} mm, every trial")

    # What each flow rate draws, and for how long. This is the table that says
    # whether the flow-rate axis is one thing or two: at a fixed volume the
    # aspiration time runs down the column, at a fixed time the volume runs up it.
    table = volume_table(rig, flow_rates)
    print(f"\naspirate {rig.aspirate_mode}; return at {rig.return_flow:g} ul/s:")
    for f, v, t_asp, t_ret in table:
        print(f"  {f:6g} ul/s -> {v:6g} ul   aspirate {t_asp:5.2f} s   "
              f"return {t_ret:5.2f} s")
    largest = max(v for _, v, _, _ in table)
    print(f"  largest volume {largest:g} ul of a {TIP_CAPACITY_UL:g} ul tip "
          f"({100 * largest / TIP_CAPACITY_UL:.0f} %)")
    check_volumes(rig, flow_rates)

    plan = height_plan(offsets)
    print(f"\nheights: {describe_heights(plan)}")
    if plan["mode"] == "adaptive":
        # every height from the floor to the ceiling, for every flow rate: an
        # upper bound, since a flow rate ends on the first height that scores
        # nothing and a later one starts where an earlier one was still winning
        n_heights = int(round((plan["max_mm"] - plan["start_mm"])
                              / plan["step_mm"])) + 1
    else:
        n_heights = len(plan["offsets"])
    n_trials = n_heights * len(flow_rates) * trials
    # Three detections, three settles, the aspiration and the return averaged
    # over the grid, and about eight seconds of gantry motion per trial. The
    # motion figure is a rough bench estimate, not a measurement.
    liquid = sum(t_asp + t_ret for _, _, t_asp, t_ret in table) / len(table)
    per_trial = 3 * t_detect + 3 * rig.settle_s + liquid + 8.0
    print(f"\nat most {n_trials} trials, about {per_trial:.0f} s each "
          f"-> {n_trials * per_trial / 3600:.1f} h "
          + ("(an upper bound: each flow rate ends on the first height that "
             "scores nothing, and starts where the last one was still winning)"
             if plan["mode"] == "adaptive" else
             "(less, where early stopping cuts a flow rate short)"))

    if rig.recorder is not None:
        worst = estimate_clip_bytes(rig.under_cam, rig.cfg, n_trials)
        free = shutil.disk_usage(str(rig.clip_dir)).free
        print(f"clips: worst case, with every trial kept, about "
              f"{worst / 1e9:.1f} GB; {free / 1e9:.1f} GB free")
        require(free > worst * 1.2,
                f"not enough room for the worst case: {worst / 1e9:.1f} GB "
                f"needed, {free / 1e9:.1f} GB free")

    return {"diameter_microns": float(obj.diameter_microns),
            "area_px": float(obj.area), "n_boxes": int(len(boxes)),
            "t_detect_s": t_detect}

## 8. One trial

`run_trial` returns one row of the table. The order:

1. frame from the top camera, detection, largest object inside the ROI;
2. that pixel into robot coordinates through `pmap`, plus the profile's pipette
   offset;
3. down to `(x, y, dish_bottom + offset)` with `min_z_height`;
4. `aspirate_in_place(volume, flow_rate)` at the grid's flow rate, with the
   volume from `Rig.volume_for` — fixed, or `flow_rate × ASPIRATE_TIME_S`;
5. lift by `lift_mm`, back to the observation pose, `SETTLE_S`;
6. check frame. **Empty means the cuboid was picked up.**
7. to the taught dish centre at `dish_bottom + RETURN_OFFSET_MM`,
   `dispense_in_place(volume, RETURN_FLOW_RATE)` — the same volume that was
   drawn — lift, park, pause;
8. confirmation frame. **The cuboid has to be back.** If it is not, the trial
   does not give up yet: it waits `RETURN_DIP_WAIT_S`, dips the tip at the
   return point — same place, same height, no dispense — parks, and looks
   again. A cuboid that stayed in the tip is usually held there by surface
   tension once the volume has gone, and touching the medium frees it.
   `return_dips` in the table says whether that was needed. Only a cuboid
   still missing after the dip is `lost`, and only then is the operator called.

The volume is aspirated and returned in every case, a `miss` included: even when
no cuboid came along there is medium in the tip, and it has to go back.

Step 7 is fixed in all three coordinates and in the flow rate, and none of them
is the parameter under test. Putting the cuboid back where it was picked up from
would let it walk across the dish over a few hundred attempts — out of the depth
of medium, the part of the frame and the local pixel scale the run began in — and
dispensing at the height being tested would make the trip back a function of that
height. `DEPOSIT_LIQUID_BACK` in the picking session does return to the pickup
point, which is right there: it is putting a missed cuboid back where it came
from, not keeping a dish still across a grid.

In [ ]:
def run_trial(rig, *, flow_rate, pickup_offset_mm, trial_index, cell_index,
              replicate) -> dict:
    """One pickup, one check, one return, one confirmation. Returns the row."""
    c = rig.cfg
    c.pickup_offset = float(pickup_offset_mm)   # pickup_height follows from it
    z = c.pickup_height
    lift = c.lift_mm
    volume = rig.volume_for(flow_rate)          # fixed, or flow x time

    row = {k: "" for k in COLUMNS}
    row.update(trial_index=trial_index, cell_index=cell_index,
               flow_rate=float(flow_rate), volume_ul=round(volume, 3),
               pickup_offset_mm=float(pickup_offset_mm), replicate=replicate,
               target_z=round(z, 3),
               timestamp=datetime.now().isoformat(timespec="seconds"))

    # 1. the dish as it is before the pickup
    park(rig)
    _, df, gantry = look(rig)
    obj = largest(df)
    if obj is None:
        raise TrialError(
            "no cuboid in the dish before the pickup. The experiment needs "
            "exactly one; put it back and run the cell again.")
    uv = (float(obj.cX), float(obj.cY))
    row.update(diameter_microns=round(float(obj.diameter_microns), 1),
               area_px=round(float(obj.area), 1),
               cX=round(uv[0], 1), cY=round(uv[1], 1))

    # 2. pixel -> robot, with the pose read next to the frame
    if not rig.pmap.covers(*uv):
        raise TrialError(f"the cuboid is at ({uv[0]:.0f}, {uv[1]:.0f}), outside "
                         f"the calibrated area of the map")
    x, y = np.asarray(rig.pmap.to_robot(uv[0], uv[1], gantry)) + rig.offset
    row.update(target_x=round(float(x), 3), target_y=round(float(y), 3))

    # 3-5. the pickup, into the ring buffer
    mark_target(rig, uv, gantry)
    if rig.recorder is not None:
        rig.recorder.start()
    move_to(rig.robot, (x, y, z + lift), min_z_height=c.dish_bottom,
            force_direct=True)
    move_to(rig.robot, (x, y, z), min_z_height=c.dish_bottom, force_direct=True)
    require_ok(rig.robot.aspirate_in_place(volume=volume,
                                           flow_rate=float(flow_rate)),
               "aspirate")
    move_relative(rig.robot, "z", lift)
    park(rig)

    # 6. the check frame: an empty dish means the cuboid is in the tip
    _, after, _ = look(rig)
    picked = largest(after) is None
    if rig.recorder is not None and rig.recorder.recording:
        rig.recorder.stop()      # the frames stay buffered until the outcome

    # 7. the return: the taught centre, at the fixed height, at the fixed flow.
    # None of the three is the parameter under test, so the trip back is the same
    # every trial and the cuboid does not wander away from where it started.
    cx, cy = rig.dish_center
    zr = c.dish_bottom + rig.return_offset
    move_to(rig.robot, (cx, cy, zr + lift), min_z_height=c.dish_bottom,
            force_direct=True)
    move_to(rig.robot, (cx, cy, zr), min_z_height=c.dish_bottom,
            force_direct=True)
    require_ok(rig.robot.dispense_in_place(volume=volume,
                                           flow_rate=float(rig.return_flow)),
               "dispense back to the dish")
    move_relative(rig.robot, "z", lift)
    park(rig)

    # 8. the confirmation frame: the cuboid has to be back
    _, back, _ = look(rig)
    returned = largest(back) is not None

    # 8b. Not back: more often than not it is still in the tip, held there by
    # surface tension after the volume has left, and a moment plus a touch of
    # the liquid lets it go. So wait, then dip the tip at the very point the
    # return went to, with no dispense - there is nothing left to push with, and
    # it is the contact with the medium that frees it - and look once more.
    # Only what is still missing after that is a loss and a call for a person.
    dips = 0
    if not returned:
        time.sleep(rig.dip_wait_s)
        move_to(rig.robot, (cx, cy, zr + lift), min_z_height=c.dish_bottom,
                force_direct=True)
        move_to(rig.robot, (cx, cy, zr), min_z_height=c.dish_bottom,
                force_direct=True)
        move_relative(rig.robot, "z", lift)
        park(rig)
        _, back, _ = look(rig)
        returned = largest(back) is not None
        dips = 1
    row["return_dips"] = dips

    after_lift = ("after the lift the dish was empty" if picked else
                  "after the lift the cuboid was still there")
    if not returned:
        row["outcome"] = "lost"
        row["note"] = f"still absent after a dip; {after_lift}"
    else:
        row["outcome"] = "success" if picked else "miss"
        if dips:
            row["note"] = "came out on the dip"
    row["clip_path"] = save_clip(rig, row)
    return row


def save_clip(rig, row: dict) -> str:
    """Write the buffered clip if the rule says to keep it. Returns its path."""
    if rig.recorder is None or not rig.recorder.frames:
        return ""
    keep = rig.keep_clip or should_keep_clip
    if not keep(row):
        return ""
    name = f"trial_{int(row['trial_index']):04d}_{row['outcome']}.mp4"
    path = Path(rig.clip_dir) / name
    try:
        rig.recorder.save_async(str(path), color=True)
    except Exception as exc:                    # a clip is never critical
        rig.log(f"    clip not saved: {exc}")
        return ""
    return f"clips/{name}"

## 9. The operator

Two things stop the run and ask for a person: a `lost` cuboid, and zero successes
at the lowest height. The first is answered from a window rather than from
`input()`, so the answer is given with the dish on screen. The window must have
focus for the keys to arrive.

In [ ]:
def operator_gate(camera, message: str, *, window: str = "pickup grid") -> bool:
    """Show the dish live and wait. Space or r carries on, Esc stops the run."""
    print(f"\n!! {message}")
    print("   space / r = carry on,  Esc = stop the run")
    lines = ["OPERATOR",
             *[message[i:i + 58] for i in range(0, len(message), 58)],
             "space/r = carry on   Esc = stop"]
    view = FrameWindow(window)
    try:
        while True:
            ok, frame = camera.read()
            if not ok:
                cv2.waitKey(20)
                continue
            vis = frame.copy()
            for i, line in enumerate(lines):
                cv2.putText(vis, line, (30, 70 + i * 55),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 0, 255), 3)
            view.show(vis)
            key = cv2.waitKey(20) & 0xFF
            if key in (ord("r"), ord(" ")):
                return True
            if key == 27:
                return False
    finally:
        view.close()

## 10. The grid

The outer loop is over flow rates, the inner one over heights **in increasing
order**. The order is not shuffled: early stopping needs an ordered traversal, and
that is a trade made on purpose.

**Early stopping.** If a flow rate scores zero successes at some height, the
remaining, higher heights for that flow rate are skipped. The next flow rate is
unaffected and starts again from the lowest height.

**Skipped cells go into the table** with the status `skipped` and no trials. A
missing row would mean "not tested" and "the data were lost" at the same time.

**Early stopping does not fire at the first height.** 0.5 is a known working point
from the previous work, so a failure there means the calibration is wrong and not
that the range has ended. That stops the run and calls the operator.

**A `lost` trial** is written with the status `lost`, is left out of the success
rate, and stops the loop until the operator confirms. It does consume one of the
`TRIALS_PER_CELL` replicates: it is an attempt that happened, and numbering it
otherwise would make the table disagree with its own row count. The success rate
is therefore `success / (success + miss)` and can rest on fewer than ten attempts,
which the analysis shows as `n` on every cell.

**Adaptive heights** (`ADAPTIVE_HEIGHTS = True`). With the volume growing with
the flow rate, starting every flow rate at 0.5 mm re-measures what is already
known, and the top of the range is not known at all. So the heights are found as
the run goes: the first flow rate starts at the floor (`PICKUP_OFFSET_START_MM`),
every later one starts at the highest height the previous flow rate was fully
successful at (every scored attempt a success; `lost` does not count against it)
or, failing that, where the previous one started; from there the walk climbs by
`PICKUP_STEP_MM` until a height scores nothing, and that ends the flow rate.
There is no upper list. If the inherited start itself scores nothing, the
heights above it are taken as failing too and the walk turns downward until a
height with successes marks the boundary; zero at the floor is still the
calibration case and stops the run. `PICKUP_OFFSET_MAX_MM` is a safety net, not
an end point: above it the tip leaves the medium and draws air; reaching it is
logged and that flow rate ends. Heights between the floor and a flow rate's
start are written as `skipped` with the reason, so every height from the floor
up to the highest one tested has a row; above the last tested height there are
none, because the range is open there. The two modes write different `grid`
sections in `run.json`, so switching modes means a new folder.

**Carrying on** starts from the first unfinished cell, and inside it from the
replicates already recorded. The grid and the dish bottom in `run.json` have to
match the constants at the top; a disagreement is refused rather than quietly
continued.

In [ ]:
def cell_key(flow, offset) -> tuple:
    return (round(float(flow), 6), round(float(offset), 6))


def read_progress(run_dir) -> dict:
    """Rows already recorded, keyed by (flow rate, height)."""
    done: dict = {}
    for row in read_trials(run_dir):
        if not row.get("flow_rate"):
            continue
        done.setdefault(cell_key(row["flow_rate"], row["pickup_offset_mm"]),
                        []).append(row)
    return done


def next_trial_index(run_dir) -> int:
    indices = [int(r["trial_index"]) for r in read_trials(run_dir)
               if r.get("trial_index")]
    return max(indices) + 1 if indices else 1


def n_successes(rows) -> int:
    return sum(1 for r in rows if r["outcome"] == "success")


def n_scored(rows) -> int:
    return sum(1 for r in rows if r["outcome"] in ("success", "miss"))


def describe_progress(run_dir, *, offsets, flow_rates, trials,
                      heights=None) -> dict:
    """What is in the folder, and where a run would carry on from.

    Walks the same plan `run_grid` would (see `make_plan`, defined with it),
    over the rows on disk, and stops at the first cell that is not finished.
    So what it names is exactly where the run picks up, in either height mode.
    """
    done = read_progress(run_dir)
    rows = read_trials(run_dir)
    if not rows:
        print(f"{Path(run_dir).name}: empty, this is a fresh start")
        return {"resuming": False}

    counts = {o: sum(1 for r in rows if r["outcome"] == o) for o in OUTCOMES}
    print(f"{Path(run_dir).name}: {len(rows)} rows — "
          + ", ".join(f"{k} {v}" for k, v in counts.items() if v))

    plan = height_plan(offsets, heights)
    print(f"  heights: {describe_heights(plan)}")
    try:
        for kind, flow, offset, _ in make_plan(plan, done, offsets=offsets,
                                               flow_rates=flow_rates,
                                               trials=trials,
                                               log=lambda *a: None):
            cell = done.get(cell_key(flow, offset), [])
            if kind == "run" and not cell_complete(cell, trials):
                print(f"  would carry on at flow {flow} ul/s, height {offset} "
                      f"mm, replicate {len(cell) + 1} of {trials}")
                return {"resuming": True, "flow": flow, "offset": offset,
                        "replicate": len(cell) + 1}
    except GridAborted as exc:
        print(f"  would stop: {exc}")
        return {"resuming": True, "stopped": str(exc)}
    print("  every cell is complete; there is nothing to carry on with")
    return {"resuming": True, "complete": True}

In [ ]:
def height_plan(offsets, heights=None) -> dict:
    """How the heights of a run are chosen, as a plain dict.

    `heights` given: used as it is (the dry run passes one). Otherwise from the
    constants, read when this is called: `ADAPTIVE_HEIGHTS` chooses between the
    listed offsets and a walk that starts at the floor, climbs by the step and
    ends where a height scores nothing. The cells that call `run_grid`,
    `preflight` and `describe_progress` pass `offsets` and nothing else, so the
    mode has to come from the constants - the same way TIP_CAPACITY_UL does.
    """
    if heights is not None:
        return dict(heights)
    if ADAPTIVE_HEIGHTS:
        return {"mode": "adaptive", "start_mm": float(PICKUP_OFFSET_START_MM),
                "step_mm": float(PICKUP_STEP_MM),
                "max_mm": float(PICKUP_OFFSET_MAX_MM)}
    return {"mode": "fixed", "offsets": [float(o) for o in offsets]}


def describe_heights(plan) -> str:
    if plan["mode"] == "adaptive":
        return (f"adaptive from {plan['start_mm']:g} mm by {plan['step_mm']:g} mm"
                f", ceiling {plan['max_mm']:g} mm")
    return f"fixed: {plan['offsets']}"


def run_conditions(rig, *, offsets, flow_rates, trials, heights=None) -> dict:
    """The conditions these rows are being taken under, as run.json states them.

    One function, used twice: `run_snapshot` folds it into the file it writes,
    and `run_grid` compares it with what the file already says. That is what
    makes a forgotten rebuild loud. Change a constant, run the grid cell but not
    the cell that builds the rig, and the trials would otherwise be appended to a
    folder whose run.json describes the previous parameters - with nothing in the
    table to tell the two halves apart.

    The fixed grid is written exactly as it was before adaptive heights existed,
    so a folder started under it still matches. The adaptive grid has its own
    shape, and the two never compare equal: a change of mode is a change of
    experiment and gets a new folder.
    """
    plan = height_plan(offsets, heights)
    flows = [float(f) for f in flow_rates]
    if plan["mode"] == "adaptive":
        grid = {"mode": "adaptive", "start_mm": plan["start_mm"],
                "step_mm": plan["step_mm"], "max_mm": plan["max_mm"],
                "flow_rates": flows, "trials_per_cell": int(trials)}
    else:
        grid = {"pickup_offsets_mm": plan["offsets"],
                "flow_rates": flows, "trials_per_cell": int(trials)}
    return {"grid": grid,
            "dish_bottom": rig.cfg.dish_bottom,
            # None when the volume follows the flow rate. The time is written
            # only in that mode: at a fixed volume it plays no part, and writing
            # it would refuse to carry on every folder recorded before it existed.
            "volume_ul": rig.volume,
            "aspirate_time_s": (rig.aspirate_time_s if rig.volume is None
                                else None),
            "return_flow_rate": rig.return_flow,
            "return_offset_mm": rig.return_offset}








def volume_table(rig, flow_rates) -> list:
    """(flow, volume, aspiration time, return time) for every flow rate."""
    return [(float(f), rig.volume_for(f), rig.volume_for(f) / float(f),
             rig.volume_for(f) / float(rig.return_flow)) for f in flow_rates]


def check_volumes(rig, flow_rates) -> None:
    """Refuse a grid whose largest volume does not fit the tip.

    With the volume following the flow rate, the top of the grid decides how
    much is drawn: 200 ul/s at one second is the whole of a 200 ul tip. Said
    before anything moves, naming the flow rate that does not fit.
    """
    too_big = [(f, v) for f, v, _, _ in volume_table(rig, flow_rates)
               if v > TIP_CAPACITY_UL]
    if too_big:
        f, v = too_big[0]
        raise GridAborted(
            f"{v:g} ul at {f:g} ul/s does not fit the tip "
            f"(TIP_CAPACITY_UL = {TIP_CAPACITY_UL:g}). Aspirating "
            f"{rig.aspirate_mode}: lower the top flow rate, ASPIRATE_TIME_S, "
            f"or set a fixed ASPIRATE_VOLUME.")


def check_conditions(rig, run_dir, *, offsets, flow_rates, trials,
                     heights=None) -> None:
    """Refuse to add rows to a folder whose run.json describes something else."""
    path = Path(run_dir) / "run.json"
    if not path.exists():
        return
    stored = json.loads(path.read_text(encoding="utf-8"))
    now = run_conditions(rig, offsets=offsets, flow_rates=flow_rates,
                         trials=trials, heights=heights)
    differences = [f"  {key}: run.json has {stored.get(key)!r}, rig has {value!r}"
                   for key, value in now.items() if stored.get(key) != value]
    if differences:
        raise GridAborted("\n".join([
            "the rig and the grid do not match this folder's run.json:",
            *differences,
            "Nothing has been written. Run the constants cell and then "
            "'Start, or carry on': that builds a rig from the numbers as they "
            "are now, and opens a folder for them."]))


def cell_complete(rows, trials) -> bool:
    """A cell is finished when it holds its trials, or a skipped row."""
    return any(r["outcome"] == "skipped" for r in rows) or len(rows) >= trials


def fully_successful(rows) -> bool:
    """Every scored attempt a success, and at least one of them.

    A `lost` is not a miss, so it does not spoil a height; a cell of nothing but
    losses says nothing about the height and does not count either way.
    """
    return n_scored(rows) > 0 and n_successes(rows) == n_scored(rows)


def _floor_abort(offset, scored) -> GridAborted:
    return GridAborted(
        f"0 successes out of {scored} at the lowest height ({offset} mm), "
        f"which is a known working point. That points at the calibration - the "
        f"dish bottom, the pipette offset, or the tip - and not at the edge of "
        f"the range. Stopping. Check the calibration and start a new run "
        f"folder; do not re-measure the dish bottom into this one.")


def fixed_plan(done, *, offsets, flow_rates, trials, log=print):
    """The listed heights, ascending, for every flow rate.

    A plan is a generator of requests, ("run", flow, offset, note) and
    ("skip", flow, offset, note), that decides each next step from `done` - the
    rows already on disk, which `run_grid` extends after every cell. That is
    what makes the walk the same whether it is running live, being resumed, or
    being described: it is a function of the table and nothing else. The
    generator reads `done` only after a yield, so it always sees the cell it
    asked for finished.

    Zero successes at a height skips the higher heights of that flow rate;
    zero at the first height is a calibration problem and stops the run.
    """
    for flow in flow_rates:
        stopped_at = None
        for i, offset in enumerate(sorted(float(o) for o in offsets)):
            rows = done.get(cell_key(flow, offset), [])
            if stopped_at is not None:
                yield ("skip", flow, offset,
                       f"early stop: 0 successes at {stopped_at} mm")
                continue
            if any(r["outcome"] == "skipped" for r in rows):
                stopped_at = offset            # written by an earlier session
                yield ("skip", flow, offset, "")
                continue
            yield ("run", flow, offset, "")
            rows = done.get(cell_key(flow, offset), [])
            if n_successes(rows) == 0 and n_scored(rows) > 0:
                if i == 0:
                    raise _floor_abort(offset, n_scored(rows))
                stopped_at = offset
                log(f"  early stop: no successes at {offset} mm, so the higher "
                    f"points for {flow} ul/s are skipped")


def adaptive_plan(done, *, flow_rates, trials, start, step, ceiling,
                  log=print):
    """Heights found as the run goes, one flow rate informing the next.

    The first flow rate starts at the floor. Every later one starts at the
    highest height the previous flow rate was fully successful at - below that
    it is not going to learn anything - or, when there was no such height,
    where the previous one started. From the start the walk climbs by `step`
    until a height scores nothing, which ends that flow rate; there is no list
    to run out of, so the top of the range is whatever the dish says it is.

    If the inherited start itself scores nothing, the higher heights are taken
    as failing too (the same reading of a zero the early stop rests on) and the
    walk turns downward, one step at a time, until a height with successes
    shows where the boundary is. Zero at the floor is the calibration case and
    stops the run, as in the fixed plan.

    `ceiling` is a safety net and not part of the experiment: above it the tip
    is out of the medium and draws air. Reaching it is logged, and that flow
    rate simply ends.

    When a flow rate is finished, the heights between the floor and its start
    that were never run are written as `skipped`, with the reason. Every height
    from the floor up to the highest one tested therefore has a row; above the
    last tested height there are none, because the range is open there.
    """
    def at(x):
        return round(x, 3)

    floor = at(start)
    prev_flow, prev_start = None, None
    for flow in flow_rates:
        if prev_flow is None:
            h0, why = floor, "the floor"
        else:
            pf = cell_key(prev_flow, 0)[0]
            wins = {o: rows for (f, o), rows in done.items()
                    if f == pf and fully_successful(rows)}
            if wins:
                h0 = at(max(wins))
                rows = wins[max(wins)]
                why = (f"{n_successes(rows)}/{n_scored(rows)} successful at "
                       f"{h0} mm, {prev_flow} ul/s")
            else:
                h0 = prev_start
                why = (f"no fully successful height at {prev_flow} ul/s; "
                       f"starting where it did")
        log(f"[{flow} ul/s] starts at {h0} mm: {why}")

        cur, first, walk_down = h0, True, False
        while True:
            yield ("run", flow, cur, why if first else "")
            rows = done.get(cell_key(flow, cur), [])
            scored = n_scored(rows)
            if scored > 0 and n_successes(rows) == 0:
                if first and cur > floor + 1e-9:
                    walk_down = True
                elif first:
                    raise _floor_abort(cur, scored)
                else:
                    log(f"  [{flow} ul/s] no successes at {cur} mm: this flow "
                        f"rate ends here")
                break
            first = False
            nxt = at(cur + step)
            if nxt > ceiling + 1e-9:
                log(f"  [{flow} ul/s] ceiling {ceiling} mm reached with "
                    f"successes still coming; ending this flow rate here")
                break
            cur = nxt

        if walk_down:
            log(f"  [{flow} ul/s] nothing at the start height {h0} mm; the "
                f"heights above are taken as failing too, walking down to "
                f"find the boundary")
            cur = at(h0 - step)
            while cur >= floor - 1e-9:
                yield ("run", flow, cur, "walking down from a start that "
                                         "scored nothing")
                rows = done.get(cell_key(flow, cur), [])
                if n_successes(rows) > 0:
                    break
                if abs(cur - floor) < 1e-9:
                    raise _floor_abort(cur, n_scored(rows))
                cur = at(cur - step)

        cur = floor
        while cur < h0 - 1e-9:
            if not done.get(cell_key(flow, cur)):
                yield ("skip", flow, cur,
                       f"not run: started at {h0} mm ({why})")
            cur = at(cur + step)
        prev_flow, prev_start = flow, h0


def make_plan(plan, done, *, offsets, flow_rates, trials, log=print):
    """The request generator for a height plan."""
    if plan["mode"] == "adaptive":
        return adaptive_plan(done, flow_rates=flow_rates, trials=trials,
                             start=plan["start_mm"], step=plan["step_mm"],
                             ceiling=plan["max_mm"], log=log)
    return fixed_plan(done, offsets=plan["offsets"], flow_rates=flow_rates,
                      trials=trials, log=log)


def run_grid(rig, run_dir, *, offsets, flow_rates, trials, gate=None,
             confirmed: bool = False, log=print, heights=None):
    """Walk the grid, writing one row per trial. Returns the results folder."""
    run_dir = Path(run_dir)
    plan = height_plan(offsets, heights)
    gate = gate or (lambda message: operator_gate(rig.camera, message))
    check_volumes(rig, flow_rates)
    check_conditions(rig, run_dir, offsets=offsets, flow_rates=flow_rates,
                     trials=trials, heights=plan)
    done = read_progress(run_dir)

    if done and not confirmed:
        raise GridAborted(
            f"{run_dir.name} already holds trials. Read what describe_progress() "
            f"printed, check that this is the same dish and the same cuboid, and "
            f"call again with confirmed=True. Carrying on is a deliberate act: "
            f"nothing in software can tell a new cuboid from the old one.")

    trial_index = next_trial_index(run_dir)
    cell_index = 0
    log(f"heights: {describe_heights(plan)}")

    for kind, flow, offset, note in make_plan(plan, done, offsets=offsets,
                                              flow_rates=flow_rates,
                                              trials=trials, log=log):
        cell_index += 1
        key = cell_key(flow, offset)
        rows = list(done.get(key, []))

        if kind == "skip":
            if not rows:
                skipped = {k: "" for k in COLUMNS}
                skipped.update(
                    cell_index=cell_index, flow_rate=float(flow),
                    pickup_offset_mm=float(offset), replicate=0,
                    outcome="skipped",
                    timestamp=datetime.now().isoformat(timespec="seconds"),
                    note=note)
                append_row(run_dir, skipped)
                done[key] = [skipped]
                log(f"[{flow} ul/s, {offset} mm] skipped: {note}")
            else:
                log(f"[{flow} ul/s, {offset} mm] skipped, from the file")
            continue

        if any(r["outcome"] == "skipped" for r in rows):
            log(f"[{flow} ul/s, {offset} mm] skipped, from the file")
            continue
        if note and len(rows) < trials:
            log(f"[{flow} ul/s, {offset} mm] {note}")

        while len(rows) < trials:
            replicate = len(rows) + 1
            row = run_trial(rig, flow_rate=flow, pickup_offset_mm=offset,
                            trial_index=trial_index, cell_index=cell_index,
                            replicate=replicate)
            append_row(run_dir, row)
            rows.append(row)
            done[key] = rows
            trial_index += 1
            log(f"[{flow} ul/s, {offset} mm] {replicate}/{trials} "
                f"{row['outcome']}"
                + (f"  ({row['note']})" if row["note"] else ""))
            if row["outcome"] == "lost":
                if not gate("the cuboid is not in the dish after the "
                            "return. Find it, put it back in the dish, "
                            "then carry on."):
                    raise GridAborted("stopped by the operator after a loss")

        successes, scored = n_successes(rows), n_scored(rows)
        log(f"[{flow} ul/s, {offset} mm] {successes}/{scored} successful"
            + (f", {len(rows) - scored} lost" if scored < len(rows) else ""))

    write_xlsx(run_dir)
    log(f"\ndone. {run_dir}")
    return run_dir

### Start, or carry on

`RESUME_FOLDER = None` starts a new folder. To continue an interrupted run, put
its folder name there, read what `describe_progress` says, and set
`CONFIRMED = True`.

This cell is also how a change of parameters takes effect. `make_rig()` reads the
constants, the taught dish, the pipette offset and the pixel map at the moment it
is called, and prints in one line what the rig will do: check the volume, the
flow and the dish bottom in that line against what you meant. A new folder gets a
new `run.json`, so two sets of parameters never land in one table; carrying an
existing folder on with different numbers is refused rather than mixed.

In [ ]:
openapi.toggle_lights()

In [ ]:
# Teach it. Use load_dish() instead when the dish has not moved since last time.
dish_pose = teach_dish()
# dish_pose = load_dish()

In [ ]:
# if "observe_2" not in profile.positions:
#     jog("bring the crosshair disc under the camera, then Enter")
#     profile.remember("observe_2", xyz(openapi))
# print("observe_2:", profile.where("observe_2"))

In [ ]:
openapi.retract_axis("leftZ")

In [ ]:
EXPERIMENT_LABEL   = "round3_1s_tconst"        # goes into the results folder name
# PICKUP_OFFSETS_MM  = [0.5, 1.5]
# PICKUP_OFFSETS_MM  = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1, 2.2, 2.3, 2.4, 2.5]
PICKUP_OFFSETS_MM  = [1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1, 2.2, 2.3, 2.4, 2.5]
ADAPTIVE_HEIGHTS       = True   # heights found as the run goes (below); False = PICKUP_OFFSETS_MM as listed
PICKUP_OFFSET_START_MM = 0.3    # the floor: the first flow rate starts here, none goes below
PICKUP_STEP_MM         = 0.1    # mm between heights
PICKUP_OFFSET_MAX_MM   = 3.0    # a safety ceiling, not an end: above it the tip is out of the medium


FLOW_RATES         = [25, 50, 75, 100, 125, 150, 175]   # ul/s, aspiration. Edit per task.
# FLOW_RATES         = [100, 125, 150, 175]   # ul/s, aspiration. Edit per task.

TRIALS_PER_CELL    = 10
RETURN_FLOW_RATE   = 50.0                # ul/s, putting the cuboid back. NOT from the grid.
RETURN_OFFSET_MM   = 1.0                 # mm above the dish bottom, at the centre. NOT from the grid.
RETURN_DIP_WAIT_S  = 10.0                 # s to wait, then dip the tip, when the cuboid did not come out
ASPIRATE_VOLUME    = None                # ul. A number = this volume at every flow rate.
                                         # None = the volume follows the flow rate (next line).
ASPIRATE_TIME_S    = 1.0                 # s. With ASPIRATE_VOLUME None: volume = flow_rate x this,
                                         # so every aspiration lasts the same time. 1.0 = "volume equals flow".
SETTLE_S           = 0.3             # pause before a check frame
RECORD_CLIPS       = False               # lower-camera recording

# The rest of what a trial needs. These are the experiment's own numbers: nothing
# here is read from picking.json and nothing is written back to it. The dish
# bottom is not among them — it comes from the pose taught with the tip.
LIFT_MM            = 20.0                # clearance above the pickup height
ROI_CENTER         = (1296, 972)         # the working circle, in pixels
ROI_RADIUS         = 600
MIN_DISTANCE_MM    = 2.0                 # here it only grows the ROI mask
CLIP_MAX_FRAMES    = 600                 # bound on the clip ring buffer
TIP_CAPACITY_UL    = 200.0               # vwr_96_tiprack_200ul_xl; every cell's volume must fit

# Running this cell applies nothing on its own: the numbers are read when the rig
# is built. Change one here, run this cell, then run "Start, or carry on" below.
_aspirate = (f"{ASPIRATE_VOLUME:g} ul at every flow" if ASPIRATE_VOLUME is not None
             else f"{ASPIRATE_TIME_S:g} s x flow (volume = flow rate x {ASPIRATE_TIME_S:g})")
_heights = (f"adaptive from {PICKUP_OFFSET_START_MM:g} by {PICKUP_STEP_MM:g} mm, "
            f"ceiling {PICKUP_OFFSET_MAX_MM:g}" if ADAPTIVE_HEIGHTS
            else f"fixed {PICKUP_OFFSETS_MM}")
print(f"heights {_heights}")
print(f"aspirate {_aspirate} | return {RETURN_FLOW_RATE:g} ul/s at "
      f"+{RETURN_OFFSET_MM:g} mm | settle {SETTLE_S:g} s | lift {LIFT_MM:g} mm | "
      f"ROI r={ROI_RADIUS} px about {ROI_CENTER}")

In [ ]:
print(f"pickup height at the lowest offset "
      f"({min(PICKUP_OFFSETS_MM)} mm): "
      f"{dish_pose[2] + min(PICKUP_OFFSETS_MM):.2f} mm")
print(f"the cuboid goes back to ({dish_pose[0]:.2f}, {dish_pose[1]:.2f}) "
      f"at {dish_pose[2] + RETURN_OFFSET_MM:.2f} mm, every time")

openapi.retract_axis('leftZ')

In [ ]:
# Derived from the constants above, printed so the run is described in one place.
N_CELLS = len(FLOW_RATES) * len(PICKUP_OFFSETS_MM)
CUBOID_SIZE_UM = (lambda m: int(m.group(1)) if m else None)(
    re.search(r"(\d+)\s*u?m", EXPERIMENT_LABEL))

print(f"{len(FLOW_RATES)} flow rates x {len(PICKUP_OFFSETS_MM)} heights "
      f"= {N_CELLS} cells, {TRIALS_PER_CELL} trials each "
      f"= {N_CELLS * TRIALS_PER_CELL} trials at most")
print("cuboid size, read off the label:",
      f"{CUBOID_SIZE_UM} um" if CUBOID_SIZE_UM else
      "not stated — run.json will record only the label")

In [ ]:
RESUME_FOLDER =  False
CONFIRMED = False           # True only to carry an existing folder on

for old in previous_runs():
    print("existing folder for this label:", old.name)

run_dir = open_run_dir(RESUME_FOLDER)
rig = make_rig()
if RECORD_CLIPS:
    require(under_cam is not None,
            "RECORD_CLIPS is on but the lower camera is not open; re-run the "
            "camera cell")
    attach_recorder(rig, under_cam, run_dir,
                    homography=profile.calibration.homography)

snapshot = run_snapshot(rig, label=EXPERIMENT_LABEL, offsets=PICKUP_OFFSETS_MM,
                        flow_rates=FLOW_RATES, trials=TRIALS_PER_CELL,
                        weights=CUBOID_WEIGHTS)
check_snapshot(run_dir, snapshot)
print("results:", run_dir)
describe_progress(run_dir, offsets=PICKUP_OFFSETS_MM, flow_rates=FLOW_RATES,
                  trials=TRIALS_PER_CELL)

In [ ]:
openapi.toggle_lights()

In [ ]:
jog()

In [ ]:
openapi.aspirate_in_place(volume = 20)

In [ ]:
openapi.home_robot()

In [ ]:
openapi.dispense_in_place(20)

In [ ]:
openapi.retract_axis('leftZ')

In [ ]:
info = preflight(rig, offsets=PICKUP_OFFSETS_MM, flow_rates=FLOW_RATES,
                 trials=TRIALS_PER_CELL)

In [ ]:
openapi.dispense_in_place(100)

In [ ]:
try:
    run_grid(rig, run_dir, offsets=PICKUP_OFFSETS_MM, flow_rates=FLOW_RATES,
             trials=TRIALS_PER_CELL, confirmed=CONFIRMED)
except GridAborted as exc:
    print(f"\nSTOPPED: {exc}")
finally:
    if rig.recorder is not None:
        rig.under_cam.detach(rig.recorder)
        rig.recorder = None
    openapi.retract_axis("leftZ")

In [ ]:
openapi.aspirate_in_place(volume = 100, flow_rate=100.0)

In [ ]:
# The xlsx on demand, from whatever the CSV holds at this moment.
print(write_xlsx(run_dir))

## 11. Analysis

Works on the CSV alone and needs nothing from the run above, so a finished folder
can be read in a fresh kernel.

Ten attempts buy less than they look like they do: the standard error of a
proportion is about 0.15 at p ≈ 0.5. A cell at 0.2 is distinguishable from one at
0.8. A cell at 0.5 is not distinguishable from one at 0.7, whatever the colours
suggest.

In [ ]:
import matplotlib.pyplot as plt

ANALYSE = run_dir           # or RESULTS_ROOT / "20260901_141233_dead_400um"

trials = pd.read_csv(Path(ANALYSE) / "trials.csv")
print(trials["outcome"].value_counts().to_string())
print("\nrun.json:")
print(json.dumps(json.loads((Path(ANALYSE) / "run.json").read_text(
    encoding="utf-8")), indent=2)[:1200])

In [ ]:
# cell_stats and rate_table are defined in section 5, next to the writing.
stats = cell_stats(trials)
table = rate_table(stats)
print(table.round(2).to_string())
stats.round(3)

In [ ]:
# The heat map. A skipped cell is not a zero, so it is grey and labelled: the two
# must not be read off the same colour.
fig, ax = plt.subplots(figsize=(1.4 * len(table.columns) + 3,
                                0.5 * len(table.index) + 3))
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("0.85")

img = ax.imshow(np.ma.masked_invalid(table.values.astype(float)), cmap=cmap,
                vmin=0, vmax=1, aspect="auto", origin="lower")
ax.set_xticks(range(len(table.columns)), [f"{c:g}" for c in table.columns])
ax.set_yticks(range(len(table.index)), [f"{i:g}" for i in table.index])
ax.set_xlabel("aspirate flow rate, ul/s")
ax.set_ylabel("pickup offset above the dish bottom, mm")
ax.set_title(f"success rate — {Path(ANALYSE).name}")

lookup = stats.set_index(["pickup_offset_mm", "flow_rate"])
for yi, offset in enumerate(table.index):
    for xi, flow in enumerate(table.columns):
        if (offset, flow) not in lookup.index:
            ax.text(xi, yi, "-", ha="center", va="center", color="0.4")
            continue
        row = lookup.loc[(offset, flow)]
        if bool(row.skipped):
            ax.text(xi, yi, "skip", ha="center", va="center", color="0.35",
                    fontsize=9, style="italic")
        elif row.n_scored == 0:
            ax.text(xi, yi, "n=0", ha="center", va="center", color="0.35")
        else:
            ax.text(xi, yi, f"{row.rate:.1f}\nn={int(row.n_scored)}",
                    ha="center", va="center", fontsize=9,
                    color="white" if row.rate < 0.6 else "black")
fig.colorbar(img, ax=ax, label="share of successful pickups")
fig.tight_layout()
plt.show()

In [ ]:
# Success against height, one line per flow rate, with Wilson intervals. The bars
# are wide on purpose: that is what ten attempts per cell actually knows.
fig, ax = plt.subplots(figsize=(9, 5.5))
for flow, g in stats[~stats.skipped].groupby("flow_rate"):
    g = g[g.n_scored > 0].sort_values("pickup_offset_mm")
    if g.empty:
        continue
    err = np.vstack([g.rate - g.ci_low, g.ci_high - g.rate])
    ax.errorbar(g.pickup_offset_mm, g.rate, yerr=err, marker="o", capsize=3,
                label=f"{flow:g} ul/s")
ax.set_xlabel("pickup offset above the dish bottom, mm")
ax.set_ylabel("share of successful pickups")
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)
ax.legend(title="aspirate flow")
ax.set_title("success against height, Wilson 95 % intervals")
fig.tight_layout()
plt.show()

In [ ]:
# Did the cuboid itself change over the run? Worth looking at even for a dead one:
# a drift here means the object measured at the end was not the one at the start.
seen = trials[trials.outcome.isin(("success", "miss", "lost"))].dropna(
    subset=["trial_index", "diameter_microns"])

fig, ax = plt.subplots(figsize=(10, 4.5))
for outcome, colour in (("success", "tab:green"), ("miss", "tab:orange"),
                        ("lost", "tab:red")):
    g = seen[seen.outcome == outcome]
    if len(g):
        ax.scatter(g.trial_index, g.diameter_microns, s=18, alpha=0.85,
                   c=colour, label=outcome)
if len(seen) > 1:
    median = seen.diameter_microns.median()
    ax.axhline(median, color="0.5", lw=1)
    ax.text(seen.trial_index.min(), median, f" median {median:.0f} um",
            va="bottom", color="0.35")
ax.set_xlabel("trial index")
ax.set_ylabel("diameter, um")
ax.set_title("the cuboid across the run")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

print(seen.diameter_microns.describe().round(1).to_string())

In [ ]:
# --- Dry run on mocks -------------------------------------------------------
from micropick.config.schema import PickingConfig
from micropick.config.schema import PixelMap as PixelMapConfig
from micropick.hardware.mock import MockRobot, open_mock_camera

MW, MH = 900, 700                       # mock frame
MK = 0.05                               # mm per pixel of the toy linear map
M_OBSERVE = np.array([150.0, 150.0, 80.0])
M_OFFSET = np.array([1.0, -2.0])        # pipette offset
M_BOTTOM = 66.4
M_CATCH_Z = M_BOTTOM + 0.6              # the tip only catches below this height
M_CENTER = M_OBSERVE[:2] + M_OFFSET     # the taught centre: pixel (MW/2, MH/2)
M_START_UV = (420.0, 330.0)             # where the cuboid begins, off centre


def mock_pixel_map() -> PixelMap:
    """to_robot(u, v, g) = g + MK * (u - cu, v - cv)."""
    s = max(MW, MH) / 2.0
    return PixelMap(PixelMapConfig(
        degree=1, cu=MW / 2, cv=MH / 2, s=s,
        coef=[[0.0, MK * s], [MK * s, 0.0]], zero=[0.0, 0.0],
        ref=[MW / 2, MH / 2], bounds=[0.0, 0.0, float(MW), float(MH)],
        image_size=[MW, MH], sweep_z=0.0))


class MockDish:
    """One cuboid, drawn as a bright square unless it is in the tip or gone."""

    def __init__(self, uv=M_START_UV, half: int = 18):
        self.uv = np.array(uv, dtype=float)
        self.half = half
        self.present = True
        self.held = False

    @property
    def deck(self) -> np.ndarray:
        return (M_OBSERVE[:2] + MK * (self.uv - np.array([MW / 2, MH / 2]))
                + M_OFFSET)

    def put_at(self, deck_xy) -> None:
        """Where a dispense leaves the cuboid: the inverse of `deck`."""
        self.uv = ((np.asarray(deck_xy, dtype=float) - M_OBSERVE[:2] - M_OFFSET)
                   / MK + np.array([MW / 2, MH / 2]))

    def render(self, i):
        frame = np.full((MH, MW, 3), 20, np.uint8)
        if self.present and not self.held:
            u, v = int(self.uv[0]), int(self.uv[1])
            cv2.rectangle(frame, (u - self.half, v - self.half),
                          (u + self.half, v + self.half), (220, 220, 220), -1)
        return frame


class _Arr:
    def __init__(self, a):
        self._a = np.asarray(a, dtype=float)

    def cpu(self):
        return self

    def numpy(self):
        return self._a


class _Boxes:
    def __init__(self, xyxy, conf):
        self.xyxy, self.conf = _Arr(xyxy), _Arr(conf)


class _Result:
    def __init__(self, boxes):
        self.boxes = boxes


class MockYOLO:
    """Boxes the cuboid when it is visible. Not a model: a stand-in for one."""

    def __init__(self, dish: MockDish):
        self.dish = dish

    def predict(self, source, **kw):
        if not (self.dish.present and not self.dish.held):
            return [_Result(_Boxes(np.zeros((0, 4)), np.zeros((0,))))]
        u, v = self.dish.uv
        h = self.dish.half + 2
        return [_Result(_Boxes([[u - h, v - h, u + h, v + h]], [0.9]))]


class MockGridRobot(MockRobot):
    """Catches the cuboid below M_CATCH_Z; drops or keeps it on chosen returns.

    `lose_on` returns lose the cuboid outright. `stick_on` returns leave it in
    the tip: the dispense goes through but the cuboid stays `held`, and it only
    comes out when the tip next touches the return height near the centre — the
    dip — and then only if `release_on_dip`. That is the bench failure the dip
    exists for, and the one it cannot fix, side by side.
    """

    def __init__(self, dish: MockDish, lose_on=(), stick_on=(),
                 release_on_dip: bool = True, catch_offset=None):
        super().__init__(position=tuple(M_OBSERVE), noise_mm=0.0,
                         speed_mm_s=4000.0)
        self.dish = dish
        self.lose_on = set(lose_on)
        self.stick_on = set(stick_on)
        self.release_on_dip = release_on_dip
        # how high above the bottom the tip still catches: one number, or a
        # dict by flow rate, so a faster flow can reach higher - which is
        # what the adaptive walk is built around
        self.catch_offset = catch_offset
        self.stuck = False
        self.returns = 0
        self.dips = 0

    def move_to_coordinates(self, coordinates, min_z_height=None,
                            force_direct=False, speed=None, verbose=True):
        super().move_to_coordinates(coordinates, min_z_height, force_direct,
                                    speed, verbose)
        # a tip holding a stuck cuboid, back down at the return height by the
        # centre, is a dip; count it, and let go if this cuboid will let go
        if self.stuck and self._pos[2] <= M_BOTTOM + 1.0 + 1e-6                 and float(np.linalg.norm(self._pos[:2] - M_CENTER)) < 0.5:
            self.dips += 1
            if self.release_on_dip:
                self.stuck = False
                self.dish.held = False
                self.dish.put_at(self._pos[:2])

    def aspirate_in_place(self, volume, flow_rate, verbose=False):
        super().aspirate_in_place(volume, flow_rate, verbose)
        near = float(np.linalg.norm(self._pos[:2] - self.dish.deck)) < 1.0
        co = self.catch_offset
        if isinstance(co, dict):
            co = co.get(float(flow_rate), co.get(int(flow_rate)))
        limit = M_CATCH_Z if co is None else M_BOTTOM + float(co)
        if near and self._pos[2] <= limit + 1e-9 and self.dish.present:
            self.dish.held = True

    def dispense_in_place(self, volume, flow_rate, verbose=False):
        super().dispense_in_place(volume, flow_rate, verbose)
        if self.dish.held:
            self.returns += 1
            if self.returns in self.stick_on:
                self.stuck = True               # the volume left, the cuboid did not
                return
            self.dish.held = False
            self.dish.put_at(self._pos[:2])     # it lands under the tip
            if self.returns in self.lose_on:
                self.dish.present = False       # never made it back to the dish


def mock_config() -> PickingConfig:
    return PickingConfig(dish_bottom=M_BOTTOM, vol=10.0, lift_mm=5.0,
                         circle_center=(MW // 2, MH // 2), circle_radius=300,
                         minimum_distance=0.0, clip_max_frames=30)


def mock_rig(dish: MockDish, robot, camera) -> Rig:
    cfg_m = mock_config()
    mmpp = MK
    return Rig(robot=robot, camera=camera, detector=MockYOLO(dish), cfg=cfg_m,
               pmap=mock_pixel_map(), offset=M_OFFSET, observe=M_OBSERVE,
               dish_center=M_CENTER, one_d_ratio=mmpp, size_ratio=mmpp * mmpp,
               settle_s=0.02, volume=10.0, aspirate_time_s=None,
               return_flow=50.0, return_offset=1.0, dip_wait_s=0.0,
               log=lambda *a: None)

In [ ]:
# --- the checks -------------------------------------------------------------
M_OFFSETS = [0.5, 0.7, 0.9]      # only 0.5 is below M_CATCH_Z
M_FLOWS = [10, 50]
M_TRIALS = 2

dry_dir = RESULTS_ROOT / "_dry_run"
if dry_dir.exists():
    shutil.rmtree(dry_dir)
dry_dir.mkdir(parents=True)
(dry_dir / "clips").mkdir()

dish = MockDish()
robot = MockGridRobot(dish, lose_on={2})        # the second return drops it
camera = open_mock_camera(width=MW, height=MH, fps=120.0, render=dish.render)
rig_m = mock_rig(dish, robot, camera)

gate_calls = []


def mock_gate(message: str) -> bool:
    """The operator finds the cuboid and puts it back."""
    gate_calls.append(message)
    dish.present = True
    return True


# the listed heights, pinned: these checks describe the fixed plan whatever
# ADAPTIVE_HEIGHTS is set to above
M_FIXED = {"mode": "fixed", "offsets": M_OFFSETS}
M_SNAPSHOT = {"label": "_dry_run", "dish_bottom": M_BOTTOM, "volume_ul": 10.0,
              "aspirate_time_s": None,
              "return_flow_rate": 50.0, "return_offset_mm": 1.0,
              "grid": {"pickup_offsets_mm": M_OFFSETS, "flow_rates": M_FLOWS,
                       "trials_per_cell": M_TRIALS}}

try:
    check_snapshot(dry_dir, M_SNAPSHOT)
    run_grid(rig_m, dry_dir, offsets=M_OFFSETS, flow_rates=M_FLOWS,
             trials=M_TRIALS, gate=mock_gate, log=lambda *a: None,
             heights=M_FIXED)

    rows = read_trials(dry_dir)
    counts = {o: sum(1 for r in rows if r["outcome"] == o) for o in OUTCOMES}

    ok_written = verdict("the CSV was written line by line",
                         (dry_dir / "trials.csv").exists() and len(rows) == 10,
                         expected="10 rows", got=f"{len(rows)} rows")

    ok_status = verdict(
        "statuses are assigned",
        counts == {"success": 3, "miss": 4, "lost": 1, "skipped": 2},
        expected={"success": 3, "miss": 4, "lost": 1, "skipped": 2},
        got=counts,
        note="0.5 mm is below the catch height and works; 0.7 and 0.9 do not; "
             "one return drops the cuboid")

    ok_gate = verdict("a loss calls the operator", len(gate_calls) == 1,
                      expected="1 call", got=f"{len(gate_calls)} calls")

    stopped = {(float(r["flow_rate"]), float(r["pickup_offset_mm"]))
               for r in rows if r["outcome"] == "skipped"}
    ok_early = verdict(
        "early stopping fires, and only above the failing height",
        stopped == {(10.0, 0.9), (50.0, 0.9)},
        expected="0.9 mm skipped for both flow rates, 0.7 mm tested",
        got=sorted(stopped))

    ok_skipped_rows = verdict(
        "skipped cells are in the table, with no trials",
        all(r["replicate"] == "0" and r["note"].startswith("early stop")
            for r in rows if r["outcome"] == "skipped"),
        expected="replicate 0 and a reason on every skipped row",
        got=[(r["replicate"], r["note"]) for r in rows
             if r["outcome"] == "skipped"])

    at_centre = float(np.hypot(*(dish.uv - np.array([MW / 2, MH / 2]))))
    ok_centre = verdict(
        "the cuboid is put back at the taught centre, not where it came from",
        at_centre < 1.0,
        expected=f"({MW / 2:.0f}, {MH / 2:.0f}) px, the taught centre",
        got=f"({dish.uv[0]:.1f}, {dish.uv[1]:.1f}) px, having started at "
            f"{M_START_UV}")

    # --- carrying on ---------------------------------------------------------
    # Drop everything for the second flow rate, as an interruption would leave
    # it, and run again: the missing cells should come back and nothing already
    # recorded should be repeated.
    kept = [r for r in rows if float(r["flow_rate"]) != 50.0]
    (dry_dir / "trials.csv").unlink()
    for r in kept:
        append_row(dry_dir, r)
    before = {int(r["trial_index"]) for r in kept if r["trial_index"]}

    run_grid(rig_m, dry_dir, offsets=M_OFFSETS, flow_rates=M_FLOWS,
             trials=M_TRIALS, gate=mock_gate, confirmed=True,
             log=lambda *a: None, heights=M_FIXED)

    resumed = read_trials(dry_dir)
    indices = [int(r["trial_index"]) for r in resumed if r["trial_index"]]
    ok_resume = verdict(
        "carrying on picks up at the first unfinished cell",
        len(resumed) == 10 and len(indices) == len(set(indices))
        and min(set(indices) - before) > max(before),
        expected="10 rows again, unique trial indices, new ones after the old",
        got=f"{len(resumed)} rows, indices {sorted(indices)}")

    ok_refuse = False
    try:
        check_snapshot(dry_dir, {**M_SNAPSHOT, "dish_bottom": M_BOTTOM + 1.0})
    except GridAborted:
        ok_refuse = True
    verdict("a changed dish bottom is refused, not carried on with", ok_refuse,
            expected="GridAborted", got="accepted" if not ok_refuse else
            "refused")

    xlsx = write_xlsx(dry_dir)
    sheets = pd.ExcelFile(xlsx).sheet_names
    ok_xlsx = verdict("the xlsx is built from the CSV",
                      xlsx.exists() and "trials" in sheets,
                      expected="a trials sheet", got=sheets)

    verdict("the fixed plan writes the grid into run.json exactly as before",
            run_conditions(rig_m, offsets=M_OFFSETS, flow_rates=M_FLOWS,
                           trials=M_TRIALS, heights=M_FIXED)["grid"]
            == M_SNAPSHOT["grid"],
            expected=M_SNAPSHOT["grid"],
            got=run_conditions(rig_m, offsets=M_OFFSETS, flow_rates=M_FLOWS,
                               trials=M_TRIALS, heights=M_FIXED)["grid"])


    # --- the volume rule -----------------------------------------------------
    # The same rig with the volume following the flow rate: one second's worth
    # at 99 ul/s is 99 ul, drawn at 99 and given back at the fixed return flow.
    from dataclasses import replace as _replace
    timed = _replace(rig_m, volume=None, aspirate_time_s=1.0)
    timed_row = run_trial(timed, flow_rate=99, pickup_offset_mm=0.5,
                          trial_index=903, cell_index=0, replicate=1)
    asp = [c for c in robot.calls if c[0] == "aspirate_in_place"][-1]
    disp = [c for c in robot.calls if c[0] == "dispense_in_place"][-1]
    verdict("with ASPIRATE_VOLUME None the volume is flow x time, and it is "
            "what the robot is told",
            asp[1:] == (99.0, 99.0) and disp[1:] == (99.0, 50.0)
            and timed_row["volume_ul"] == 99.0,
            expected="aspirate(99, 99), dispense(99, 50), volume_ul 99",
            got=f"aspirate{asp[1:]}, dispense{disp[1:]}, "
                f"volume_ul {timed_row['volume_ul']}")

    ok_cap = False
    try:
        check_volumes(timed, [10, TIP_CAPACITY_UL + 1])
    except GridAborted as exc:
        ok_cap = f"{TIP_CAPACITY_UL + 1:g} ul/s" in str(exc)
    verdict("a grid whose volume does not fit the tip is refused before it moves",
            ok_cap, expected=f"GridAborted naming {TIP_CAPACITY_UL + 1:g} ul/s",
            got="refused" if ok_cap else "accepted")

    # a folder written before volume_ul existed must not take new rows
    old_dir = RESULTS_ROOT / "_dry_run_old_columns"
    if old_dir.exists():
        shutil.rmtree(old_dir)
    old_dir.mkdir(parents=True)
    with open(old_dir / "trials.csv", "w", newline="", encoding="utf-8") as fh:
        csv.writer(fh).writerow([c for c in COLUMNS if c != "volume_ul"])
    ok_header = False
    try:
        append_row(old_dir, timed_row)
    except GridAborted as exc:
        ok_header = "older column set" in str(exc)
    verdict("rows are not appended under an older CSV header",
            ok_header and (old_dir / "trials.csv").read_text().count(chr(10)) == 1,
            expected="GridAborted, file still one line",
            got="refused" if ok_header else "appended")
    shutil.rmtree(old_dir)

    # --- the dip before a loss ----------------------------------------------
    # A cuboid that stays in the tip after the dispense. It has to come out on
    # the dip and score as an ordinary success with the dip on record, with no
    # second dispense and no operator; one that will not come out is a loss, and
    # only then. The robots share the dish, so each starts with it in place.
    def fresh_dish():
        dish.present, dish.held = True, False

    fresh_dish()
    sticky = MockGridRobot(dish, stick_on={1})
    r_sticky = _replace(rig_m, robot=sticky)
    stuck_row = run_trial(r_sticky, flow_rate=10, pickup_offset_mm=0.5,
                          trial_index=904, cell_index=0, replicate=1)
    n_disp = sum(1 for c in sticky.calls if c[0] == "dispense_in_place")
    verdict("a cuboid stuck in the tip comes out on the dip and is not a loss",
            stuck_row["outcome"] == "success" and stuck_row["return_dips"] == 1
            and stuck_row["note"] == "came out on the dip"
            and sticky.dips == 1 and n_disp == 1,
            expected="success, return_dips 1, one dip, one dispense",
            got=(stuck_row["outcome"], stuck_row["return_dips"],
                 stuck_row["note"], f"dips {sticky.dips}",
                 f"dispenses {n_disp}"))

    fresh_dish()
    stuck_fast = MockGridRobot(dish, stick_on={1}, release_on_dip=False)
    lost_row = run_trial(_replace(rig_m, robot=stuck_fast), flow_rate=10,
                         pickup_offset_mm=0.5, trial_index=905, cell_index=0,
                         replicate=1)
    verdict("a cuboid that will not come out is a loss, after the dip",
            lost_row["outcome"] == "lost" and lost_row["return_dips"] == 1
            and lost_row["note"].startswith("still absent after a dip")
            and stuck_fast.dips == 1,
            expected="lost, return_dips 1, one dip, then the operator",
            got=(lost_row["outcome"], lost_row["return_dips"], lost_row["note"]))

    fresh_dish()
    plain_row = run_trial(rig_m, flow_rate=10, pickup_offset_mm=0.5,
                          trial_index=906, cell_index=0, replicate=1)
    verdict("an ordinary return records no dip",
            plain_row["outcome"] == "success" and plain_row["return_dips"] == 0
            and plain_row["note"] == "",
            expected="success, return_dips 0",
            got=(plain_row["outcome"], plain_row["return_dips"]))

    # --- adaptive heights ----------------------------------------------------
    # A tip that catches higher at the faster flow. The first flow rate climbs
    # from the floor to its first zero; the second starts where the first was
    # still fully successful, climbs to its own zero, and the height it never
    # ran is written as skipped with the reason.
    M_ADAPT = {"mode": "adaptive", "start_mm": 0.5, "step_mm": 0.1,
               "max_mm": 1.5}

    def adaptive_run(flows, catch, *, folder, max_mm=1.5, gate=None):
        d = RESULTS_ROOT / folder
        if d.exists():
            shutil.rmtree(d)
        (d / "clips").mkdir(parents=True)
        fresh_dish()
        bot = MockGridRobot(dish, catch_offset=catch)
        r = _replace(rig_m, robot=bot)
        plan = {**M_ADAPT, "max_mm": max_mm}
        check_snapshot(d, {**M_SNAPSHOT, "label": folder,
                           **run_conditions(r, offsets=M_OFFSETS, flow_rates=flows,
                                            trials=M_TRIALS, heights=plan)})
        run_grid(r, d, offsets=M_OFFSETS, flow_rates=flows, trials=M_TRIALS,
                 gate=gate or mock_gate, log=lambda *a: None, heights=plan)
        return d, r

    def cells_of(d):
        """{flow: {offset: outcome-summary}} from the CSV, for the assertions."""
        out = {}
        for r in read_trials(d):
            f, o = float(r["flow_rate"]), float(r["pickup_offset_mm"])
            out.setdefault(f, {}).setdefault(o, []).append(r["outcome"])
        return out

    d1, _ = adaptive_run([10, 50], {10.0: 0.6, 50.0: 0.8}, folder="_dry_adaptive")
    c = cells_of(d1)
    verdict("adaptive: the first flow rate climbs from the floor to its first zero",
            sorted(c[10.0]) == [0.5, 0.6, 0.7]
            and c[10.0][0.5] == ["success"] * 2 and c[10.0][0.7] == ["miss"] * 2,
            expected="0.5, 0.6 successful, 0.7 all misses, nothing above",
            got={o: v for o, v in sorted(c[10.0].items())})
    verdict("adaptive: the next flow rate starts where the last was still fully "
            "successful, and the height below is skipped with the reason",
            sorted(c[50.0]) == [0.5, 0.6, 0.7, 0.8, 0.9]
            and c[50.0][0.5] == ["skipped"] and c[50.0][0.6] == ["success"] * 2
            and c[50.0][0.9] == ["miss"] * 2
            and next(r["note"] for r in read_trials(d1)
                     if r["outcome"] == "skipped").startswith("not run: started at 0.6 mm"),
            expected="0.5 skipped ('not run: started at 0.6 mm ...'), 0.6-0.8 "
                     "successful, 0.9 all misses, nothing above",
            got={o: v for o, v in sorted(c[50.0].items())})

    # carrying on: drop the second flow rate and run again
    rows1 = read_trials(d1)
    kept1 = [r for r in rows1 if float(r["flow_rate"]) != 50.0]
    (d1 / "trials.csv").unlink()
    for r in kept1:
        append_row(d1, r)
    fresh_dish()
    r_again = _replace(rig_m, robot=MockGridRobot(dish, catch_offset={10.0: 0.6, 50.0: 0.8}))
    said = describe_progress(d1, offsets=M_OFFSETS, flow_rates=[10, 50],
                             trials=M_TRIALS, heights=M_ADAPT)
    verdict("adaptive: describe_progress names the inherited start of the next flow rate",
            said.get("flow") == 50 and said.get("offset") == 0.6,
            expected="flow 50, height 0.6", got=said)
    run_grid(r_again, d1, offsets=M_OFFSETS, flow_rates=[10, 50], trials=M_TRIALS,
             gate=mock_gate, confirmed=True, log=lambda *a: None, heights=M_ADAPT)
    rows2 = read_trials(d1)
    same = [(r["flow_rate"], r["pickup_offset_mm"], r["outcome"]) for r in rows1] \
        == [(r["flow_rate"], r["pickup_offset_mm"], r["outcome"]) for r in rows2]
    idx = [int(r["trial_index"]) for r in rows2 if r["trial_index"]]
    verdict("adaptive: carrying on rebuilds the same table, indices unique",
            same and len(idx) == len(set(idx)),
            expected="same cells and outcomes, unique indices",
            got=f"same {same}, {len(idx)} indices, {len(set(idx))} unique")

    # walking down: the slower flow rate inherits a start it cannot manage
    d2, _ = adaptive_run([50, 10], {50.0: 0.8, 10.0: 0.6}, folder="_dry_adaptive_down")
    c = cells_of(d2)
    verdict("adaptive: nothing at the inherited start walks down to the boundary "
            "and never above it",
            sorted(c[10.0]) == [0.5, 0.6, 0.7, 0.8]
            and c[10.0][0.8] == ["miss"] * 2 and c[10.0][0.7] == ["miss"] * 2
            and c[10.0][0.6] == ["success"] * 2 and c[10.0][0.5] == ["skipped"],
            expected="0.8 and 0.7 all misses, 0.6 successful, 0.5 skipped, nothing above 0.8",
            got={o: v for o, v in sorted(c[10.0].items())})

    # the floor: a tip that never catches is the calibration case
    floor_hit = False
    try:
        adaptive_run([10], 0.0, folder="_dry_adaptive_floor")
    except GridAborted as exc:
        floor_hit = "calibration" in str(exc)
    verdict("adaptive: zero at the floor stops the run and names the calibration",
            floor_hit and sorted(cells_of(RESULTS_ROOT / "_dry_adaptive_floor").get(10.0, {})) == [0.5],
            expected="GridAborted, only 0.5 in the table", got=floor_hit)

    # the ceiling: a tip that always catches ends at the safety net, no abort
    d3, _ = adaptive_run([10], 9.0, folder="_dry_adaptive_ceiling", max_mm=0.7)
    c = cells_of(d3)
    verdict("adaptive: the ceiling ends a flow rate that keeps winning, without an abort",
            sorted(c[10.0]) == [0.5, 0.6, 0.7]
            and all(v == ["success"] * 2 for v in c[10.0].values()),
            expected="0.5, 0.6, 0.7 successful and nothing else",
            got={o: v for o, v in sorted(c[10.0].items())})

    for folder in ("_dry_adaptive", "_dry_adaptive_down", "_dry_adaptive_floor",
                   "_dry_adaptive_ceiling"):
        if (RESULTS_ROOT / folder).exists():
            shutil.rmtree(RESULTS_ROOT / folder)
    # --- the clip decision ---------------------------------------------------
    # Whether a clip is kept, not whether it encodes: the codec belongs to the
    # machine. Two trials with a recorder attached, one that works and one that
    # does not, and the buffer has to be dropped for the first and kept for the
    # second.
    attach_recorder(rig_m, camera, dry_dir)     # no homography, so no ROI box
    try:
        won = run_trial(rig_m, flow_rate=10, pickup_offset_mm=0.5,
                        trial_index=901, cell_index=0, replicate=1)
        lost_it = run_trial(rig_m, flow_rate=10, pickup_offset_mm=0.9,
                            trial_index=902, cell_index=0, replicate=1)
        buffered = len(rig_m.recorder.frames)
        verdict("a successful pickup throws its clip away",
                won["outcome"] == "success" and won["clip_path"] == "",
                expected="success, no clip", got=(won["outcome"],
                                                  won["clip_path"] or "none"))
        verdict("a failed pickup keeps its clip, named for the trial",
                lost_it["outcome"] == "miss"
                and lost_it["clip_path"] == "clips/trial_0902_miss.mp4",
                expected="miss, clips/trial_0902_miss.mp4",
                got=(lost_it["outcome"], lost_it["clip_path"] or "none"))
        verdict("the ring buffer stays inside clip_max_frames",
                0 < buffered <= rig_m.cfg.clip_max_frames,
                expected=f"1..{rig_m.cfg.clip_max_frames} frames",
                got=f"{buffered} frames",
                note="encoding is asynchronous and depends on the machine's "
                     "codecs; this checks the decision, not the file")
    finally:
        camera.detach(rig_m.recorder)
        rig_m.recorder = None

    print("\n--- the table the dry run produced ---")
    print(pd.read_csv(dry_dir / "trials.csv")[
        ["trial_index", "flow_rate", "pickup_offset_mm", "replicate",
         "outcome", "note"]].to_string(index=False))
    print("\n--- success rates ---")
    print(rate_table(cell_stats(pd.read_csv(dry_dir / "trials.csv")))
          .round(2).to_string())
finally:
    camera.close()

## Flow sensor

In [ ]:
%pip install -q sensirion-uart-scc1 pandas

from serial.tools import list_ports
for p in list_ports.comports():
    print(p.device, "|", p.description)

In [ ]:
import threading, time
import numpy as np, pandas as pd
from sensirion_shdlc_driver import ShdlcSerialPort, ShdlcConnection
from sensirion_uart_scc1.scc1_shdlc_device import Scc1ShdlcDevice

PORT = "COM4"        # <-- из ячейки 1
ADDR = 0x08
START = {"water": 0x3608, "ipa": 0x3615}
STOP  = 0x3FF9
SCALE_FLOW, SCALE_TEMP = 500.0, 200.0   # ml/min = raw/500, degC = raw/200

def _crc8(data: bytes) -> int:
    crc = 0xFF
    for b in data:
        crc ^= b
        for _ in range(8):
            crc = ((crc << 1) ^ 0x31) & 0xFF if crc & 0x80 else (crc << 1) & 0xFF
    return crc

def _cmd(dev, code):
    dev.i2c_transceive(ADDR, code.to_bytes(2, "big"), 0, 100)

def _read(dev):
    raw = dev.i2c_transceive(ADDR, b"", 9, 100)
    vals = []
    for i in (0, 3, 6):
        if _crc8(raw[i:i+2]) != raw[i+2]:
            raise IOError(f"CRC error: {raw.hex()}")
        vals.append(int.from_bytes(raw[i:i+2], "big", signed=True))
    flow, temp, flags = vals
    return flow / SCALE_FLOW, temp / SCALE_TEMP, flags & 0xFFFF

def log_flow(seconds=30, hz=20, medium="water", port=PORT):
    rows = []
    with ShdlcSerialPort(port=port, baudrate=115200) as sp:
        dev = Scc1ShdlcDevice(ShdlcConnection(sp), target_address=0)
        assert dev.get_sensor_voltage() == 0, "кабель выставлен на 5 В — сенсору нужно 3.3 В!"
        dev.sensor_reset()                      # на случай, если Viewer оставил его в измерении
        time.sleep(0.05)
        _cmd(dev, START[medium])
        time.sleep(0.05)                        # прогрев ~50 мс
        try:
            _read(dev)                          # первое значение выбрасываем
        except IOError:
            pass
        t0, n, period = time.monotonic(), 0, 1.0 / hz
        try:
            while True:
                target = t0 + n * period
                time.sleep(max(0.0, target - time.monotonic()))
                n += 1
                t = time.monotonic() - t0
                try:
                    flow, temp, flags = _read(dev)
                except IOError as e:
                    print("skip:", e); continue
                rows.append({"t_s": t, "flow_ul_min": flow * 1000, "flow_ml_min": flow,
                             "temp_C": temp, "air_in_line": flags & 1,
                             "high_flow": (flags >> 1) & 1, "smoothing": (flags >> 5) & 1})
                if t >= seconds:
                    break
        except KeyboardInterrupt:
            pass
        finally:
            _cmd(dev, STOP)
    return pd.DataFrame(rows)

def _read_ul_s(dev):
    raw = dev.i2c_transceive(ADDR, None, 9, 100)
    vals = []
    for i in (0, 3, 6):
        if _crc8(raw[i:i+2]) != raw[i+2]:
            raise IOError(f"CRC error: {raw.hex()}")
        vals.append(int.from_bytes(raw[i:i+2], "big", signed=True))
    flow, temp, flags = vals
    return flow / 30.0, temp / 200.0, flags & 0xFFFF     # ul/s, degC, flags

import threading, time
import numpy as np, pandas as pd
from sensirion_shdlc_driver import ShdlcSerialPort, ShdlcConnection
from sensirion_uart_scc1.scc1_shdlc_device import Scc1ShdlcDevice
from sensirion_uart_scc1.drivers.scc1_slf3x import Scc1Slf3x
from sensirion_uart_scc1.drivers.slf_common import SlfMode


class FlowLogger:
    """SLF3S-1300F через SCC1. Кабель сэмплирует, питон вычерпывает буфер (ёмкость 40).
    Поток в мкл/с, объём — интегралом и тотализатором."""

    BUFFER_CAPACITY = 40

    def __init__(self, port="COM4", interval_ms=20, drain_s=0.2,
                 liquid=SlfMode.LIQUI_1, warmup_s=1.0):
        assert interval_ms >= 20, "проверено только на >=20 мс"
        assert drain_s < self.BUFFER_CAPACITY * interval_ms / 1000 * 0.5, "дренаж реже половины бюджета буфера"
        self.port, self.interval_ms, self.drain_s = port, interval_ms, drain_s
        self.liquid, self.warmup_s = liquid, warmup_s
        self.interval_s = interval_ms / 1000.0
        self.offset_ul_s, self.dropped, self._n_seen = 0.0, 0, 0
        self._rows, self._marks = [], []
        self._lock, self._stop = threading.Lock(), threading.Event()
        self._drain_log = []

    def __enter__(self):
        self._sp = ShdlcSerialPort(port=self.port, baudrate=115200).__enter__()
        try:
            dev = Scc1ShdlcDevice(ShdlcConnection(self._sp), target_address=0)
            assert dev.get_sensor_voltage() == 0, "кабель на 5 В — сенсору нужно 3.3 В!"
            self.s = Scc1Slf3x(dev, liquid_mode=self.liquid)
            self.scale = (self.s.get_flow_unit_and_scale() or (500, None))[0]   # тиков на ml/min

            if self.s.get_continuous_measurement_status() is not None:
                self.s.stop_continuous_measurement()
            self.s.set_totalizator_status(True)
            self.s.start_continuous_measurement(interval_ms=self.interval_ms)
            time.sleep(0.1)
            self.s.read_extended_buffer()          # выбрасываем прогревочные сэмплы
            self._t0 = time.monotonic()
            self._thread = threading.Thread(target=self._loop, daemon=True)
            self._thread.start()
        except BaseException:
            self._sp.__exit__(None, None, None)
            raise
        time.sleep(self.warmup_s)
        return self

    def __exit__(self, *exc):
        try:
            self._stop.set()
            if getattr(self, "_thread", None):
                self._thread.join(timeout=2.0)
            self._drain()
            self.s.stop_continuous_measurement()
        except Exception as e:
            print("warn на закрытии:", e)
        finally:
            self._sp.__exit__(*exc)

    def _to_ul_s(self, raw):
        return raw / self.scale * 1000.0 / 60.0

    @staticmethod
    def _flags_to_int(flags):
        if isinstance(flags, int):
            return flags
        for attr in ("value", "flags", "raw"):
            v = getattr(flags, attr, None)
            if isinstance(v, int):
                return v
        try:
            return int(flags)
        except Exception:
            return 0

    def _drain(self):
        t_call = time.monotonic() - self._t0
        try:
            dropped, _, samples = self.s.read_extended_buffer()
        except Exception as e:
            with self._lock:
                self._drain_log.append((t_call, -1, -1, repr(e)))
            return
        with self._lock:
            self._drain_log.append((t_call, len(samples), dropped, ""))
            self._n_seen += dropped
            self.dropped += dropped
            for flow, temp, flags in samples:
                self._rows.append((self._n_seen * self.interval_s,
                                   self._to_ul_s(flow), temp / 200.0, int(flags)))
                self._n_seen += 1

    def _loop(self):
        while not self._stop.is_set():
            self._stop.wait(self.drain_s)
            self._drain()

    def mark(self, label):
        t = time.monotonic() - self._t0
        with self._lock:
            self._marks.append((t, label))
        return t

    def zero(self, seconds=2.0):
        """Ноль при остановленном потоке. Достаточно одного раза на серию."""
        t_start = time.monotonic() - self._t0
        time.sleep(seconds)
        d = self.df_raw
        base = d[(d.t_s >= t_start) & (d.t_s <= t_start + seconds)]
        self.offset_ul_s = float(base.flow_ul_s.mean())
        return self.offset_ul_s

    @property
    def df_raw(self):
        with self._lock:
            rows = list(self._rows)
        return pd.DataFrame(rows, columns=["t_s", "flow_ul_s", "temp_C", "flags"])

    @property
    def df(self):
        d = self.df_raw
        if d.empty:
            return d
        d["flow_ul_s"] -= self.offset_ul_s
        d["air_in_line"] = d.flags & 1
        d["high_flow"] = (d.flags >> 1) & 1
        d["vol_ul"] = np.concatenate([[0.0], np.cumsum(np.diff(d.t_s) *
                      (d.flow_ul_s.values[:-1] + d.flow_ul_s.values[1:]) / 2)])
        return d

    @property
    def marks(self):
        with self._lock:
            return pd.DataFrame(self._marks, columns=["t_s", "label"])

    def totalizer_ul(self):
        """Объём по тотализатору кабеля: sum(raw) * interval_s / 30."""
        v = self.s.get_totalizator_value()
        return None if v is None else v * self.interval_s / 30.0

    def segment(self, start_label, end_label):
        m = self.marks
        t0 = m.loc[m.label == start_label, "t_s"].iloc[0]
        t1 = m.loc[m.label == end_label, "t_s"].iloc[0]
        seg = self.df.query("@t0 <= t_s <= @t1").reset_index(drop=True)
        return seg, float(np.trapezoid(seg.flow_ul_s, seg.t_s))

In [ ]:
with FlowLogger(interval_ms=20, drain_s=0.2) as fl:
    fl.zero(2.0)                       # поток стоит!
    fl.s.reset_totalizator()
    fl.mark("aspirate_start")
    openapi.aspirate_in_place(100,100)
    fl.mark("aspirate_end")
    time.sleep(1.0)                    # хвост, пока поток затухает
    tot = fl.totalizer_ul()

In [ ]:
log = pd.DataFrame(fl._drain_log, columns=["t_call_s", "n", "dropped", "error"])
wall = log.t_call_s.iloc[-1] - log.t_call_s.iloc[0]
got = log.n[log.n > 0].sum()
print(f"стенное время {wall:.2f} с | сэмплов {got} | эффективный интервал {wall/got*1000:.1f} мс")
print(f"ошибок дренажа: {(log.n < 0).sum()}")
print(log.head(15))
print(fl.marks)

In [ ]:
openapi.aspirate_in_place(100, 25)

In [ ]:
openapi.dispense_in_place(100, 200)

In [ ]:
import matplotlib.pyplot as plt

d = fl.df_raw.copy()
d["flow_ul_s"] = d["flow_ul_s"] - fl.offset_ul_s
m = fl.marks
t0 = m.loc[m.label == "aspirate_start", "t_s"].iloc[0]
t1 = m.loc[m.label == "aspirate_end", "t_s"].iloc[0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})

ax1.plot(d.t_s, d.flow_ul_s, lw=1.2, color="tab:blue")
ax1.axvspan(t0, t1, color="tab:orange", alpha=0.15, label="между метками")
ax1.axvline(t0, color="tab:orange", lw=1); ax1.axvline(t1, color="tab:orange", lw=1)
ax1.axhline(0, color="grey", lw=0.6)
ax1.set_ylabel("поток, мкл/с"); ax1.grid(alpha=0.3); ax1.legend(loc="upper right")
ax1.set_title(f"задано 100 мкл при 100 мкл/с | пик {d.flow_ul_s.abs().max():.1f} мкл/с")

# накопленный объём по всей записи
vol = np.concatenate([[0.0], np.cumsum(np.diff(d.t_s) *
      (d.flow_ul_s.values[:-1] + d.flow_ul_s.values[1:]) / 2)])
ax2.plot(d.t_s, np.abs(vol), lw=1.2, color="tab:green")
ax2.axvspan(t0, t1, color="tab:orange", alpha=0.15)
ax2.axhline(100, color="red", ls="--", lw=0.8, label="задано 100 мкл")
ax2.set_xlabel("время, с"); ax2.set_ylabel("объём, мкл")
ax2.grid(alpha=0.3); ax2.legend(loc="lower right")
plt.tight_layout(); plt.show()

print(f"полный интеграл по записи: {abs(vol[-1]):.1f} мкл")
print(f"только между метками:      {abs(np.trapezoid(d.query('@t0 <= t_s <= @t1').flow_ul_s, d.query('@t0 <= t_s <= @t1').t_s)):.1f} мкл")
print(f"поток на момент метки end: {d.iloc[(d.t_s - t1).abs().argmin()].flow_ul_s:.1f} мкл/с")